# CausalLOB — M2 Technical Foundation

**CSE 516 Probabilistic Graphical Models**  
**Individual project (Group G9)**

This notebook contains the M2 implementation for the original CausalLOB project on FI-2010.  The pipeline rebuilds the state representation, learns and validates a lagged graph, fits the numerical DBN, and runs probabilistic inference under partial observation.  The dataset is observational, so graph edges are interpreted as lagged dependency / causal-candidate relations under the stated assumptions rather than experimentally identified effects.


## Data rebuild and state construction

In [ ]:
# %% [markdown]
# # CausalLOB — Step 2 Automatic Data Rebuild
#
# This notebook behaves like the original M1 notebook:
#
# - mounts Google Drive automatically;
# - searches local Colab storage and Drive for the required FI-2010 files;
# - searches inside ZIP files, including nested ZIPs;
# - if the files are not found, downloads the FI-2010 Kaggle mirror automatically;
# - reconstructs the canonical 394,337-row raw dataset;
# - infers 50 day×security boundary-separated sequences;
# - creates the six leakage-safe CausalLOB state variables;
# - creates the 20-step volatility regime variable;
# - builds the lag-valid temporal checkpoint for lags 1–5;
# - saves everything to Google Drive;
# - reuses valid checkpoints on later runs.
#
# No manual upload is needed unless KaggleHub itself fails.

# %%
# ==========================================================================================
# CELL 1 — INSTALL / IMPORT / DRIVE / CONFIG
# ==========================================================================================

import sys, subprocess, importlib.util, warnings, time, json, hashlib, gc, shutil, zipfile
from pathlib import Path

REQUIRED_PACKAGES = {
    "kagglehub": "kagglehub",
    "pyarrow": "pyarrow",
}
for module_name, pip_name in REQUIRED_PACKAGES.items():
    if importlib.util.find_spec(module_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])

import numpy as np
import pandas as pd
import kagglehub

try:
    from IPython.display import display
except Exception:
    display = print

warnings.filterwarnings("ignore")

try:
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
    else:
        print("Google Drive already mounted.")
except Exception:
    pass

T0 = time.time()

# ---------------- USER SETTINGS ----------------
FORCE_REBUILD = False

BASE = Path(os.environ.get("CAUSALLOB_ROOT", "/content/drive/MyDrive/CausalLOB"))
ROOT = BASE / "STEP2_REBUILD"
CHECKPOINT_DIR = ROOT / "checkpoints"
RESULTS_DIR = ROOT / "results"
for p in [ROOT, CHECKPOINT_DIR, RESULTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

CHECKPOINT_VERSION = "STEP2_REBUILD_v1_2026-09-17"

EXPECTED_ROWS = 394_337
EXPECTED_DAYS = 10
EXPECTED_SEGMENTS_PER_DAY = 5
EXPECTED_SEQUENCES = 50

DATASET_HANDLE = "ulfricirons/fi-2010"

RAW_CKPT = CHECKPOINT_DIR / "01_canonical_raw.npz"
BOUNDARY_CKPT = CHECKPOINT_DIR / "02_boundaries.npz"
FEATURE_PARQUET = CHECKPOINT_DIR / "03_features.parquet"
FEATURE_NPZ = CHECKPOINT_DIR / "03_features.npz"
TEMPORAL_NPZ = CHECKPOINT_DIR / "04_temporal_primary.npz"
QUALITY_JSON = CHECKPOINT_DIR / "05_quality_report.json"
SOURCE_MANIFEST = CHECKPOINT_DIR / "source_manifest.json"
RUN_MANIFEST = CHECKPOINT_DIR / "RUN_MANIFEST.json"

REQUIRED_NAMES = (
    ["Train_Dst_NoAuction_DecPre_CF_1.txt"]
    + [f"Test_Dst_NoAuction_DecPre_CF_{i}.txt" for i in range(1, 10)]
)
REQUIRED_SET = set(REQUIRED_NAMES)

RAW_FEATURE_NAMES = []
for level in range(1, 11):
    RAW_FEATURE_NAMES += [
        f"ask_price_l{level}",
        f"ask_volume_l{level}",
        f"bid_price_l{level}",
        f"bid_volume_l{level}",
    ]

LABEL_NAMES = [
    "label_h10",
    "label_h20",
    "label_h30",
    "label_h50",
    "label_h100",
]

PRIMARY_CAUSAL_VARS = [
    "log_relative_spread",
    "log_total_depth_l5",
    "imbalance_l1",
    "imbalance_l2_l5",
    "quote_ofi_depth_norm",
    "log_mid_return_1",
]

REGIME_VARS = ["event_volatility_20"]

print("="*100)
print("CAUSALLOB — STEP 2 AUTOMATIC DATA REBUILD")
print("="*100)
print("Checkpoint root :", CHECKPOINT_DIR)
print("Force rebuild   :", FORCE_REBUILD)
print("Dataset handle  :", DATASET_HANDLE)

# %%
# ==========================================================================================
# CELL 2 — HELPERS + AUTOMATIC DATASET DISCOVERY / DOWNLOAD
# ==========================================================================================

def sha256_file(path, chunk_size=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            block = f.read(chunk_size)
            if not block:
                break
            h.update(block)
    return h.hexdigest()

def atomic_savez_compressed(path, **kwargs):
    path = Path(path)
    tmp = path.with_name(path.name + ".tmp.npz")
    np.savez_compressed(tmp, **kwargs)
    tmp.replace(path)

def atomic_json(path, obj):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(obj, indent=2), encoding="utf-8")
    tmp.replace(path)

def validate_version(z):
    return (
        "checkpoint_version" in z.files
        and str(z["checkpoint_version"].item()) == CHECKPOINT_VERSION
    )

def find_required_txt_files(root):
    root = Path(root)
    found = {}
    if not root.exists():
        return found
    try:
        for path in root.rglob("*.txt"):
            if path.name in REQUIRED_SET:
                found[path.name] = path
    except Exception:
        pass
    return found

def extract_required_from_zip(zip_path, output_dir, seen=None):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    extracted = {}

    if seen is None:
        seen = set()

    zip_path = Path(zip_path)
    key = str(zip_path.resolve()) if zip_path.exists() else str(zip_path)
    if key in seen:
        return extracted
    seen.add(key)

    try:
        with zipfile.ZipFile(zip_path, "r") as zf:
            members = zf.namelist()
            basename_to_member = {Path(m).name: m for m in members}

            for required_name in REQUIRED_NAMES:
                if required_name in basename_to_member:
                    destination = output_dir / required_name
                    if not destination.exists():
                        with zf.open(basename_to_member[required_name]) as src, open(destination, "wb") as dst:
                            shutil.copyfileobj(src, dst)
                    extracted[required_name] = destination

            nested = [
                m for m in members
                if m.lower().endswith(".zip")
                and ("decpre" in m.lower() or "noauction" in m.lower() or "fi2010" in m.lower())
            ]

            for member in nested:
                nested_path = output_dir / Path(member).name
                if not nested_path.exists():
                    with zf.open(member) as src, open(nested_path, "wb") as dst:
                        shutil.copyfileobj(src, dst)
                extracted.update(
                    extract_required_from_zip(nested_path, output_dir, seen=seen)
                )

    except (zipfile.BadZipFile, PermissionError, OSError):
        pass

    return extracted

def search_zip_roots(roots, extract_dir):
    found = {}
    for root in roots:
        root = Path(root)
        if not root.exists():
            continue
        try:
            for zp in root.rglob("*.zip"):
                name = zp.name.lower()
                if (
                    "fi" in name
                    or "auction" in name
                    or "decpre" in name
                    or "orderbook" in name
                ):
                    found.update(extract_required_from_zip(zp, extract_dir))
                    if len(found) == 10:
                        return found
        except Exception:
            continue
    return found

def locate_or_download_sources():
    """
    1) Search already-extracted files.
    2) Search ZIPs already present in Colab / CausalLOB Drive folders.
    3) Download Kaggle FI-2010 mirror automatically.
    4) Search/extract downloaded files.
    """
    extract_dir = Path("/content/fi2010_decpre")
    extract_dir.mkdir(parents=True, exist_ok=True)

    local_roots = [
        Path("/content/fi2010_decpre"),
        Path("/content/fi2010_download"),
        Path("/content"),
        Path("/content/drive/MyDrive/CausalLOB"),
    ]

    found = {}

    # Search direct TXT files first.
    for root in local_roots:
        found.update(find_required_txt_files(root))
        if len(found) == 10:
            print("✓ Found all 10 FI-2010 Decimal-Precision files locally.")
            return found

    print(f"Found {len(found)}/10 required TXT files directly.")

    # Search ZIP archives already present.
    zip_roots = [
        Path("/content"),
        Path("/content/drive/MyDrive/CausalLOB"),
    ]
    found.update(search_zip_roots(zip_roots, extract_dir))
    found.update(find_required_txt_files(extract_dir))

    if len(found) == 10:
        print("✓ Extracted all 10 files from an existing local/Drive ZIP.")
        return found

    # Kaggle fallback.
    print(f"Still only {len(found)}/10 files available.")
    print("Downloading FI-2010 Kaggle mirror automatically...")

    dataset_path = Path(kagglehub.dataset_download(DATASET_HANDLE))
    print("Kaggle dataset path:", dataset_path)

    found.update(find_required_txt_files(dataset_path))

    if len(found) < 10:
        found.update(search_zip_roots([dataset_path], extract_dir))

    if len(found) < 10:
        # Some Kaggle mirrors have arbitrary ZIP names; inspect all of them.
        try:
            for zp in dataset_path.rglob("*.zip"):
                found.update(extract_required_from_zip(zp, extract_dir))
                if len(found) == 10:
                    break
        except Exception:
            pass

    found.update(find_required_txt_files(extract_dir))
    found.update(find_required_txt_files(dataset_path))

    missing = REQUIRED_SET - set(found)
    if missing:
        raise FileNotFoundError(
            "Automatic acquisition failed. Missing files:\n"
            + "\n".join(sorted(missing))
            + "\n\nIf KaggleHub is unavailable, upload the FI-2010 ZIP anywhere under /content "
              "and rerun this cell; the notebook will find it automatically."
        )

    print("✓ Automatic acquisition complete.")
    return found

def load_fi2010_matrix(path):
    path = Path(path)

    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        first_line = f.readline()
    first_line_count = len(first_line.split())

    flat = np.fromfile(path, dtype=np.float32, sep=" ")
    if flat.size % 149 != 0:
        raise ValueError(
            f"{path.name}: total values {flat.size:,} not divisible by 149."
        )

    if first_line_count == 149:
        matrix = flat.reshape(-1, 149).T
    else:
        matrix = flat.reshape(149, -1)

    if matrix.shape[0] != 149:
        raise RuntimeError(f"{path.name}: unexpected matrix shape {matrix.shape}")

    return matrix

print("Automatic acquisition helpers ready.")

# %%
# ==========================================================================================
# CELL 3 — CANONICAL RAW DATASET
# ==========================================================================================

def build_canonical_raw_checkpoint():
    found = locate_or_download_sources()

    day_file_map = {
        1: found["Train_Dst_NoAuction_DecPre_CF_1.txt"]
    }
    for cf in range(1, 10):
        day_file_map[cf + 1] = found[f"Test_Dst_NoAuction_DecPre_CF_{cf}.txt"]

    X_days, Y_days, days, sample_ids = [], [], [], []
    source_rows = []

    for d in range(1, 11):
        path = day_file_map[d]
        print(f"Loading day {d}: {path}")
        M = load_fi2010_matrix(path)

        Xd = M[0:40, :].T.astype(np.float32, copy=True)
        Yd = np.rint(M[144:149, :].T).astype(np.int8)

        if Xd.shape[1] != 40:
            raise RuntimeError("Expected 40 raw LOB columns.")
        if not np.isfinite(Xd).all():
            raise RuntimeError(f"Non-finite raw values on day {d}.")
        if not np.isin(Yd, [1,2,3]).all():
            raise RuntimeError(f"Unexpected labels on day {d}: {np.unique(Yd)}")

        n = len(Xd)
        X_days.append(Xd)
        Y_days.append(Yd)
        days.append(np.full(n, d, dtype=np.int8))
        sample_ids.append(np.arange(1, n+1, dtype=np.int32))

        source_rows.append({
            "day": d,
            "file": str(path),
            "rows": int(n),
            "sha256": sha256_file(path),
        })

    X_raw = np.concatenate(X_days, axis=0)
    Y = np.concatenate(Y_days, axis=0)
    day = np.concatenate(days, axis=0)
    sample_in_day = np.concatenate(sample_ids, axis=0)

    if len(X_raw) != EXPECTED_ROWS:
        raise RuntimeError(
            f"Expected {EXPECTED_ROWS:,} rows, found {len(X_raw):,}."
        )

    atomic_savez_compressed(
        RAW_CKPT,
        checkpoint_version=np.array(CHECKPOINT_VERSION),
        X_raw=X_raw,
        Y=Y,
        day=day,
        sample_in_day=sample_in_day,
        raw_feature_names=np.array(RAW_FEATURE_NAMES),
        label_names=np.array(LABEL_NAMES),
    )
    atomic_json(SOURCE_MANIFEST, {
        "checkpoint_version": CHECKPOINT_VERSION,
        "dataset_handle": DATASET_HANDLE,
        "sources": source_rows,
    })

    return X_raw, Y, day, sample_in_day

if RAW_CKPT.exists() and not FORCE_REBUILD:
    with np.load(RAW_CKPT, allow_pickle=False) as z:
        if not validate_version(z):
            raise RuntimeError(
                "Existing raw checkpoint version mismatch. "
                "Set FORCE_REBUILD=True once."
            )
        X_raw = z["X_raw"]
        Y = z["Y"]
        day = z["day"]
        sample_in_day = z["sample_in_day"]
    print("✓ Reused raw checkpoint:", RAW_CKPT)
else:
    X_raw, Y, day, sample_in_day = build_canonical_raw_checkpoint()
    print("✓ Built raw checkpoint:", RAW_CKPT)

print("Raw shape:", X_raw.shape)
print("Labels:", Y.shape)
print("Days:", np.unique(day))

# %%
# ==========================================================================================
# CELL 4 — INFER 50 BOUNDARY-SEPARATED SEQUENCES
#
# FI-2010 concatenates five anonymous instruments inside each day.
# Boundaries are inferred from discontinuities in the raw LOB representation.
# ==========================================================================================

def robust_row_distance(X):
    """
    Distance between adjacent rows after robust coordinate scaling.
    Used only to locate large discontinuities that separate anonymous stocks.
    """
    X64 = X.astype(np.float64, copy=False)
    scale = np.median(np.abs(np.diff(X64, axis=0)), axis=0)
    scale = np.where(scale < 1e-12, 1.0, scale)

    D = np.abs(np.diff(X64, axis=0)) / scale
    return np.median(D, axis=1)

def infer_five_segments_per_day(X_raw, day):
    instrument_segment = np.empty(len(X_raw), dtype=np.int8)
    sequence_id = np.empty(len(X_raw), dtype=np.int16)
    is_segment_start = np.zeros(len(X_raw), dtype=bool)

    diagnostics = []
    seq_counter = 0

    for d in range(1, EXPECTED_DAYS+1):
        idx = np.flatnonzero(day == d)
        Xd = X_raw[idx]

        dist = robust_row_distance(Xd)

        # Four largest internal discontinuities = boundaries between five anonymous stocks.
        top4 = np.argsort(dist)[-4:]
        boundaries_local = np.sort(top4 + 1)

        starts = np.r_[0, boundaries_local]
        ends = np.r_[boundaries_local, len(idx)]

        selected = np.sort(dist[top4])
        nonselected = np.delete(dist, top4)
        ratio_min = float(selected.min() / np.median(nonselected))
        ratio_median = float(np.median(selected) / np.median(nonselected))

        for seg, (s,e) in enumerate(zip(starts, ends), start=1):
            gidx = idx[s:e]
            instrument_segment[gidx] = seg
            sequence_id[gidx] = seq_counter
            is_segment_start[gidx[0]] = True
            seq_counter += 1

        diagnostics.append({
            "day": d,
            "boundaries_local": boundaries_local.tolist(),
            "segment_lengths": (ends-starts).tolist(),
            "min_separation_ratio": ratio_min,
            "median_separation_ratio": ratio_median,
        })

    if seq_counter != EXPECTED_SEQUENCES:
        raise RuntimeError(
            f"Expected {EXPECTED_SEQUENCES} sequences, found {seq_counter}."
        )

    return instrument_segment, sequence_id, is_segment_start, diagnostics

if BOUNDARY_CKPT.exists() and not FORCE_REBUILD:
    with np.load(BOUNDARY_CKPT, allow_pickle=False) as z:
        if not validate_version(z):
            raise RuntimeError("Boundary checkpoint version mismatch.")
        instrument_segment = z["instrument_segment"]
        sequence_id = z["sequence_id"]
        is_segment_start = z["is_segment_start"]
    diagnostics = json.loads((CHECKPOINT_DIR/"02_boundary_diagnostics.json").read_text())
    print("✓ Reused boundary checkpoint.")
else:
    instrument_segment, sequence_id, is_segment_start, diagnostics = \
        infer_five_segments_per_day(X_raw, day)

    atomic_savez_compressed(
        BOUNDARY_CKPT,
        checkpoint_version=np.array(CHECKPOINT_VERSION),
        instrument_segment=instrument_segment,
        sequence_id=sequence_id,
        is_segment_start=is_segment_start,
    )
    atomic_json(CHECKPOINT_DIR/"02_boundary_diagnostics.json", diagnostics)
    print("✓ Built boundary checkpoint.")

print("Sequences:", len(np.unique(sequence_id)))
print("Min separation ratio:",
      min(x["min_separation_ratio"] for x in diagnostics))
print("Median separation ratio:",
      np.median([x["median_separation_ratio"] for x in diagnostics]))

# %%
# ==========================================================================================
# CELL 5 — BUILD LEAKAGE-SAFE MICROSTRUCTURE FEATURES
# ==========================================================================================

def sequence_safe_shift(arr, seq, lag=1):
    arr = np.asarray(arr)
    out = np.full(arr.shape, np.nan, dtype=np.float64)

    if lag == 0:
        return arr.astype(np.float64, copy=True)

    for sid in np.unique(seq):
        idx = np.flatnonzero(seq == sid)
        if len(idx) <= lag:
            continue
        out[idx[lag:]] = arr[idx[:-lag]]
    return out

def sequence_safe_rolling_std(x, seq, window=20):
    out = np.full(len(x), np.nan, dtype=np.float64)
    for sid in np.unique(seq):
        idx = np.flatnonzero(seq == sid)
        s = pd.Series(x[idx])
        out[idx] = s.rolling(window=window, min_periods=window).std(ddof=0).to_numpy()
    return out

if FEATURE_NPZ.exists() and FEATURE_PARQUET.exists() and not FORCE_REBUILD:
    with np.load(FEATURE_NPZ, allow_pickle=False) as z:
        if not validate_version(z):
            raise RuntimeError("Feature checkpoint version mismatch.")
    df = pd.read_parquet(FEATURE_PARQUET)
    print("✓ Reused feature checkpoint.")
else:
    ask_price = X_raw[:,0::4].astype(np.float64)
    ask_vol   = X_raw[:,1::4].astype(np.float64)
    bid_price = X_raw[:,2::4].astype(np.float64)
    bid_vol   = X_raw[:,3::4].astype(np.float64)

    ask1 = ask_price[:,0]
    bid1 = bid_price[:,0]
    askv1 = ask_vol[:,0]
    bidv1 = bid_vol[:,0]

    mid = 0.5*(ask1+bid1)
    spread = ask1-bid1
    relative_spread = spread/mid

    total_depth_l5 = (
        ask_vol[:,:5].sum(axis=1)
        + bid_vol[:,:5].sum(axis=1)
    )

    imbalance_l1 = (
        (bidv1-askv1)
        /(bidv1+askv1)
    )

    bid_l2_l5 = bid_vol[:,1:5].sum(axis=1)
    ask_l2_l5 = ask_vol[:,1:5].sum(axis=1)
    imbalance_l2_l5 = (
        (bid_l2_l5-ask_l2_l5)
        /(bid_l2_l5+ask_l2_l5)
    )

    bid1_prev = sequence_safe_shift(bid1, sequence_id, 1)
    ask1_prev = sequence_safe_shift(ask1, sequence_id, 1)
    bidv1_prev = sequence_safe_shift(bidv1, sequence_id, 1)
    askv1_prev = sequence_safe_shift(askv1, sequence_id, 1)
    mid_prev = sequence_safe_shift(mid, sequence_id, 1)

    bid_component = (
        (bid1 >= bid1_prev).astype(np.float64)*bidv1
        - (bid1 <= bid1_prev).astype(np.float64)*bidv1_prev
    )
    ask_component = (
        (ask1 >= ask1_prev).astype(np.float64)*askv1_prev
        - (ask1 <= ask1_prev).astype(np.float64)*askv1
    )
    quote_ofi = bid_component + ask_component

    avg_l1_depth = 0.5*((bidv1+askv1)+(bidv1_prev+askv1_prev))
    quote_ofi_depth_norm = quote_ofi / avg_l1_depth

    log_mid_return_1 = np.log(mid/mid_prev)
    event_volatility_20 = sequence_safe_rolling_std(
        log_mid_return_1,
        sequence_id,
        window=20
    )

    df = pd.DataFrame({
        "day": day,
        "sample_in_day": sample_in_day,
        "instrument_segment": instrument_segment,
        "sequence_id": sequence_id,
        "is_segment_start": is_segment_start,
        "log_relative_spread": np.log(relative_spread),
        "log_total_depth_l5": np.log(total_depth_l5),
        "imbalance_l1": imbalance_l1,
        "imbalance_l2_l5": imbalance_l2_l5,
        "quote_ofi_depth_norm": quote_ofi_depth_norm,
        "log_mid_return_1": log_mid_return_1,
        "event_volatility_20": event_volatility_20,
    })

    for j,name in enumerate(LABEL_NAMES):
        df[name] = Y[:,j]

    df.to_parquet(FEATURE_PARQUET, index=False)

    atomic_savez_compressed(
        FEATURE_NPZ,
        checkpoint_version=np.array(CHECKPOINT_VERSION),
        features=df[PRIMARY_CAUSAL_VARS+REGIME_VARS].to_numpy(np.float64),
        Y=Y,
        day=day,
        instrument_segment=instrument_segment,
        sequence_id=sequence_id,
        sample_in_day=sample_in_day,
        feature_names=np.array(PRIMARY_CAUSAL_VARS+REGIME_VARS),
        label_names=np.array(LABEL_NAMES),
    )

    print("✓ Built feature checkpoint.")

print("Feature frame:", df.shape)
print(df[PRIMARY_CAUSAL_VARS+REGIME_VARS].isna().sum())

# %%
# ==========================================================================================
# CELL 6 — TEMPORAL CHECKPOINT FOR LAGS 1–5 + FINAL QA
# ==========================================================================================

TAU_MAX_PRIMARY = 5

def sequence_safe_lag_valid_mask(seq, current_valid, tau_max):
    valid = current_valid.copy()
    for lag in range(1, tau_max+1):
        same = np.zeros(len(seq), dtype=bool)
        same[lag:] = seq[lag:] == seq[:-lag]

        lag_valid = np.zeros(len(seq), dtype=bool)
        lag_valid[lag:] = current_valid[:-lag]

        valid &= same & lag_valid
    return valid

current_valid = np.isfinite(
    df[PRIMARY_CAUSAL_VARS+REGIME_VARS].to_numpy(np.float64)
).all(axis=1)

full_lag_valid = sequence_safe_lag_valid_mask(
    sequence_id,
    current_valid,
    TAU_MAX_PRIMARY,
)

if TEMPORAL_NPZ.exists() and not FORCE_REBUILD:
    with np.load(TEMPORAL_NPZ, allow_pickle=False) as z:
        if not validate_version(z):
            raise RuntimeError("Temporal checkpoint version mismatch.")
    print("✓ Reused temporal checkpoint.")
else:
    X_current = df.loc[current_valid, PRIMARY_CAUSAL_VARS].to_numpy(np.float64)
    X_state_with_regime = df.loc[
        current_valid, PRIMARY_CAUSAL_VARS+REGIME_VARS
    ].to_numpy(np.float64)

    Y_temporal = df.loc[current_valid, LABEL_NAMES].to_numpy(np.int8)
    day_temporal = df.loc[current_valid, "day"].to_numpy(np.int8)
    segment_temporal = df.loc[current_valid, "instrument_segment"].to_numpy(np.int8)
    seq_temporal = df.loc[current_valid, "sequence_id"].to_numpy(np.int16)
    sample_temporal = df.loc[current_valid, "sample_in_day"].to_numpy(np.int32)

    lag_mask_current = full_lag_valid[current_valid]

    atomic_savez_compressed(
        TEMPORAL_NPZ,
        checkpoint_version=np.array(CHECKPOINT_VERSION),
        X_current=X_current,
        X_state_with_regime=X_state_with_regime,
        Y=Y_temporal,
        day=day_temporal,
        instrument_segment=segment_temporal,
        sequence_id=seq_temporal,
        sample_in_day=sample_temporal,
        feature_names=np.array(PRIMARY_CAUSAL_VARS),
        regime_feature_names=np.array(REGIME_VARS),
        label_names=np.array(LABEL_NAMES),
        full_lag_valid=lag_mask_current,
    )
    print("✓ Built temporal checkpoint.")

# ---------------------- QA ----------------------
n_sequences = len(np.unique(sequence_id))
n_current = int(current_valid.sum())
n_full_lag = int(full_lag_valid.sum())
warmup_invalid = len(df) - n_current

qa = {
    "checkpoint_version": CHECKPOINT_VERSION,
    "raw_rows": int(len(X_raw)),
    "raw_shape": list(X_raw.shape),
    "label_shape": list(Y.shape),
    "days": int(len(np.unique(day))),
    "boundary_separated_sequences": int(n_sequences),
    "current_valid_rows": n_current,
    "warmup_invalid_rows": int(warmup_invalid),
    "fully_t_to_t_minus_5_valid_rows": n_full_lag,
    "publisher_engineered_features_used": False,
    "primary_features": PRIMARY_CAUSAL_VARS,
    "regime_features": REGIME_VARS,
    "label_names": LABEL_NAMES,
}

# Expected values from the canonical rebuild.
checks = {
    "raw_rows_394337": len(X_raw) == EXPECTED_ROWS,
    "days_10": len(np.unique(day)) == 10,
    "sequences_50": n_sequences == 50,
    "current_valid_393337": n_current == 393_337,
    "warmups_1000": warmup_invalid == 1000,
    "full_lag_valid_393087": n_full_lag == 393_087,
    "labels_valid": bool(np.isin(Y,[1,2,3]).all()),
}

qa["checks"] = checks
qa["all_checks_pass"] = bool(all(checks.values()))

atomic_json(QUALITY_JSON, qa)
atomic_json(RUN_MANIFEST, {
    "version": CHECKPOINT_VERSION,
    "root": str(ROOT),
    "checkpoints": {
        "raw": str(RAW_CKPT),
        "boundaries": str(BOUNDARY_CKPT),
        "features_parquet": str(FEATURE_PARQUET),
        "features_npz": str(FEATURE_NPZ),
        "temporal": str(TEMPORAL_NPZ),
        "quality": str(QUALITY_JSON),
    },
})

print("="*100)
print("STEP 2 FINAL QA")
print("="*100)
print(json.dumps(qa, indent=2))

if not qa["all_checks_pass"]:
    raise RuntimeError(
        "Step 2 QA failed. Do not continue to structure learning."
    )

print("\n✓ STEP 2 COMPLETE — ALL QA CHECKS PASS")
print("✓ No manual dataset upload required.")
print("✓ Checkpoints saved to:", CHECKPOINT_DIR)
print(f"Runtime this session: {time.time()-T0:.2f} sec")

Google Drive already mounted.
CAUSALLOB — STEP 2 AUTOMATIC DATA REBUILD
Checkpoint root : /content/drive/MyDrive/CausalLOB/STEP2_REBUILD/checkpoints
Force rebuild   : False
Dataset handle  : ulfricirons/fi-2010
Automatic acquisition helpers ready.
Found 0/10 required TXT files directly.
Still only 0/10 files available.


100%|██████████| 1.95G/1.95G [01:48<00:00, 19.3MB/s]

Extracting files...


Kaggle dataset path: /root/.cache/kagglehub/datasets/ulfricirons/fi-2010/versions/1
✓ Automatic acquisition complete.
Loading day 1: /root/.cache/kagglehub/datasets/ulfricirons/fi-2010/versions/1/BenchmarkDatasets/NoAuction/3.NoAuction_DecPre/NoAuction_DecPre_Training/Train_Dst_NoAuction_DecPre_CF_1.txt
Loading day 2: /root/.cache/kagglehub/datasets/ulfricirons/fi-2010/versions/1/BenchmarkDatasets/NoAuction/3.NoAuction_DecPre/NoAuction_DecPre_Testing/Test_Dst_NoAuction_DecPre_CF_1.txt
Loading day 3: /root/.cache/kagglehub/datasets/ulfricirons/fi-2010/versions/1/BenchmarkDatasets/NoAuction/3.NoAuction_DecPre/NoAuction_DecPre_Testing/Test_Dst_NoAuction_DecPre_CF_2.txt
Loading day 4: /root/.cache/kagglehub/datasets/ulfricirons/fi-2010/versions/1/BenchmarkDatasets/NoAuction/3.NoAuction_DecPre/NoAuction_DecPre_Testing/Test_Dst_NoAuction_DecPre_CF_3.txt
Loading day 5: /root/.cache/kagglehub/datasets/ulfricirons/fi-2010/versions/1/BenchmarkDatasets/NoAuction/3.NoAuction_DecPre/NoAuction_DecPr

## Independent boundary audit

In [ ]:
import numpy as np
import pandas as pd

boundary_audit = []
total_mismatches = 0

for d in range(1, 11):
    idx = np.flatnonzero(day == d)

    # Independent reconstruction using the original M1 method
    mid = (
        X_raw[idx, 0].astype(np.float64)
        + X_raw[idx, 2].astype(np.float64)
    ) / 2.0

    jumps = np.abs(np.diff(mid))
    ranked = np.sort(jumps)[::-1]

    ratio = ranked[3] / max(ranked[4], 1e-12)

    original_cuts = np.sort(
        np.argpartition(jumps, -4)[-4:] + 1
    )

    # Boundaries selected by the new Step 2
    new_cuts = np.flatnonzero(
        np.diff(sequence_id[idx]) != 0
    ) + 1

    matches = np.array_equal(original_cuts, new_cuts)

    total_mismatches += int(not matches)

    boundary_audit.append({
        "day": d,
        "original_cuts": original_cuts.tolist(),
        "new_cuts": new_cuts.tolist(),
        "separation_ratio": float(ratio),
        "exact_match": bool(matches)
    })

audit = pd.DataFrame(boundary_audit)

display(audit)

assert np.isfinite(audit["separation_ratio"]).all(), \
    "Non-finite boundary separation!"

assert (audit["separation_ratio"] > 10).all(), \
    "Weak security-boundary separation!"

assert total_mismatches == 0, \
    "New and original boundary methods disagree!"

assert int(is_segment_start.sum()) == 50, \
    "Expected exactly 50 segment starts."

print("BOUNDARY INDEPENDENT AUDIT: PASS")

,day,original_cuts,new_cuts,separation_ratio,exact_match
0,1,"[3454, 9772, 14694, 25413]","[3454, 9772, 14694, 25413]",50.562585,True
1,2,"[5079, 11201, 17166, 23878]","[5079, 11201, 17166, 23878]",31.888876,True
2,3,"[3903, 9314, 13341, 18527]","[3903, 9314, 13341, 18527]",64.141772,True
3,4,"[2806, 9798, 15140, 25959]","[2806, 9798, 15140, 25959]",21.095325,True
4,5,"[3030, 9758, 15704, 22082]","[3030, 9758, 15704, 22082]",80.250627,True
5,6,"[2263, 8506, 14113, 21120]","[2263, 8506, 14113, 21120]",44.285916,True
6,7,"[2801, 9861, 16601, 24455]","[2801, 9861, 16601, 24455]",52.222209,True
7,8,"[2647, 11309, 19900, 33129]","[2647, 11309, 19900, 33129]",31.111102,True
8,9,"[1873, 11144, 21180, 34060]","[1873, 11144, 21180, 34060]",63.998914,True
9,10,"[1888, 7016, 12738, 18559]","[1888, 7016, 12738, 18559]",67.307318,True


BOUNDARY INDEPENDENT AUDIT: PASS


## LPCMCI structure discovery

In [ ]:
# %% [markdown]
# # CausalLOB — Step 3: Preflight and frozen structure discovery
#
#
# Run cells in order on Colab. It finds your *existing* Step 2 checkpoints in Drive; **no raw ZIP download and no preprocessing rerun**. The LPCMCI call can be expensive. Completed discovery is reused only if its SHA-256 data and configuration keys match. Nothing modifies the older M1/M2 checkpoints.
#
# **Interpretation:** LPCMCI returns observational lagged candidate relationships under its assumptions, not intervention-identified causal effects. Circle endpoints are not proven hidden confounding. ParCorr values are conditional dependence statistics, not causal effect sizes. FI-2010 lags denote representation steps, not seconds or exchange ticks. Old exploration already viewed Days 8–10; they cannot be claimed researcher-blind external data in later evaluations.

# %%
# CELL 1 — Environment and read-only inputs
import os, sys, subprocess, importlib.util, importlib.metadata
from pathlib import Path
import json, hashlib, time
import numpy as np
import pandas as pd

DESIRED_TIGRAMITE = '5.2.10.1'
try:
    installed = importlib.metadata.version('tigramite')
except importlib.metadata.PackageNotFoundError:
    installed = None
if installed != DESIRED_TIGRAMITE:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', f'tigramite=={DESIRED_TIGRAMITE}'])

from tigramite import data_processing as pp
from tigramite.lpcmci import LPCMCI
from tigramite.independence_tests.parcorr import ParCorr

try:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive')
    else:
        print('Google Drive already mounted.')
except ImportError:
    pass

ROOT=Path(os.environ.get('CAUSALLOB_ROOT','/content/drive/MyDrive/CausalLOB'))
IN=ROOT/'STEP2_REBUILD'/'checkpoints'
OUT=ROOT/'STEP3_REBUILD_FROZEN'
CKPT=OUT/'checkpoints'
RESULTS=OUT/'results'
for p in [CKPT, RESULTS]: p.mkdir(parents=True, exist_ok=True)
OLD=ROOT/'M1_G9'/'checkpoints'
INPUT_VERSION='STEP2_REBUILD_v1_2026-09-17'
STAGE_VERSION='STEP3_DISCOVERY_v1_2026-09-17'
DISCOVERY_DAYS=(1,2,3,4,5)
VALIDATION_DAYS=(6,7)
RESERVED_DAYS=(8,9,10)
FEATURES=['log_relative_spread','log_total_depth_l5','imbalance_l1',
          'imbalance_l2_l5','quote_ofi_depth_norm','log_mid_return_1']
REGIME='event_volatility_20'
LAGS=5
FORCE_DISCOVERY=False  # only set True for intentional *identical-config* rerun

for filename in ['01_canonical_raw.npz','02_boundaries.npz','03_features.parquet',
                 '04_temporal_primary.npz','source_manifest.json','05_quality_report.json']:
    if not (IN/filename).exists():
        raise FileNotFoundError(f'Step 2 checkpoint missing: {IN/filename}')

T0=time.time()
print('Input (read-only):',IN)
print('New output root  :',OUT)
print('Discovery D1–5; internal validation D6–7; historically exposed D8–10.')

# %%
# CELL 2 — Independent preflight; stop on discrepancies; no model fitting

def sha_arrays(*arrays):
    h=hashlib.sha256()
    for arr in arrays:
        arr=np.ascontiguousarray(arr)
        h.update(str(arr.shape).encode()); h.update(str(arr.dtype).encode())
        h.update(arr.view(np.uint8))
    return h.hexdigest()

def atomic_json(path, obj):
    tmp=path.with_suffix(path.suffix+'.tmp')
    tmp.write_text(json.dumps(obj,indent=2,allow_nan=False), encoding='utf-8')
    tmp.replace(path)

def chk(z, name):
    if str(z['checkpoint_version'].item()) != INPUT_VERSION:
        raise RuntimeError(f'Wrong Step 2 version in {name}')

with np.load(IN/'01_canonical_raw.npz',allow_pickle=False) as z:
    chk(z,'raw')
    raw=z['X_raw']; labels=z['Y']; raw_day=z['day']; sample=z['sample_in_day']
with np.load(IN/'02_boundaries.npz',allow_pickle=False) as z:
    chk(z,'boundaries')
    seq=z['sequence_id']; segments=z['instrument_segment']; starts=z['is_segment_start']
with np.load(IN/'04_temporal_primary.npz',allow_pickle=False) as z:
    chk(z,'temporal')
    Xt=z['X_current']; Xreg=z['X_state_with_regime']; yt=z['Y']
    dt=z['day']; st=z['sequence_id']; segt=z['instrument_segment']; pos=z['sample_in_day']
    names=z['feature_names'].astype(str).tolist()
    old_lag_mask=z['full_lag_valid']
df=pd.read_parquet(IN/'03_features.parquet')
manifest=json.loads((IN/'source_manifest.json').read_text(encoding='utf-8'))
q=json.loads((IN/'05_quality_report.json').read_text(encoding='utf-8'))

assert raw.shape==(394337,40) and labels.shape==(394337,5)
assert df.shape[0]==len(raw) and names==FEATURES
assert q.get('all_checks_pass') is True
assert np.array_equal(np.unique(raw_day),np.arange(1,11))
assert set(np.unique(labels))=={1,2,3}
assert len(np.unique(seq))==50 and np.count_nonzero(starts)==50
assert np.all((raw[:,0]>raw[:,2]) & (raw[:,0]>0) & (raw[:,2]>0))
assert np.all(raw[:,1::4]>=0) and np.all(raw[:,3::4]>=0)
assert np.all(raw[:,1]>0) and np.all(raw[:,3]>0)
assert len(manifest.get('sources',[]))==10
assert len({s['day'] for s in manifest['sources']})==10
assert all(len(s['sha256'])==64 for s in manifest['sources'])

# Full exact comparison to PREVIOUS M1 when accessible; no assumption that path exists.
original_comparison={}
for name, keys in [('01_canonical_raw.npz',['X_raw','Y','day','sample_in_day']),
                   ('02_boundaries.npz',['sequence_id','instrument_segment','is_segment_start']),
                   ('04_temporal_primary.npz',['X_current','X_state_with_regime','Y','day',
                                               'sequence_id','sample_in_day','full_lag_valid'])]:
    prev=OLD/name
    if prev.exists():
        with np.load(IN/name,allow_pickle=False) as a, np.load(prev,allow_pickle=False) as b:
            for key in keys:
                if key not in b.files: raise RuntimeError(f'Old M1 missing {name}:{key}')
                exact=bool(np.array_equal(a[key],b[key],equal_nan=True)) if np.issubdtype(a[key].dtype,np.number) else bool(np.array_equal(a[key],b[key]))
                original_comparison[f'{name}:{key}']=exact
                if not exact: raise RuntimeError(f'NEW vs original M1 mismatch: {name}:{key}')
    else:
        original_comparison[name]='OLD_CHECKPOINT_NOT_AVAILABLE'

# Independently recover boundaries from raw best-level midpoint jumps.
boundary_rows=[]
for d in range(1,11):
    ix=np.flatnonzero(raw_day==d)
    assert np.array_equal(sample[ix],np.arange(1,len(ix)+1))
    midpoint=(raw[ix,0].astype(np.float64)+raw[ix,2].astype(np.float64))/2
    jumps=np.abs(np.diff(midpoint))
    ranked=np.sort(jumps)[::-1]
    ratio=float(ranked[3]/max(ranked[4],1e-12))
    independent=np.sort(np.argpartition(jumps,-4)[-4:]+1)
    found=np.flatnonzero(np.diff(seq[ix])!=0)+1
    assert np.array_equal(independent,found) and np.isfinite(ratio) and ratio>10
    boundary_rows.append({'day':d,'fourth_to_fifth_jump_ratio':ratio,
                          'exact_cut_match':True,'cuts':independent.tolist()})
assert np.all(starts == np.r_[True, seq[1:]!=seq[:-1]])
assert np.array_equal(segments,seq%5+1)
assert np.array_equal(raw_day,seq//5+1)

# Independent full-array reconstruction of *all* 7 model/regime features.
a=raw[:,0::4].astype(np.float64); av=raw[:,1::4].astype(np.float64)
b=raw[:,2::4].astype(np.float64); bv=raw[:,3::4].astype(np.float64)
mid=(a[:,0]+b[:,0])/2
prev=np.arange(len(raw))-1
prev[starts]=-1
valid_prev=prev>=0
features_ref=np.full((len(raw),7),np.nan,dtype=np.float64)
features_ref[:,0]=np.log((a[:,0]-b[:,0])/mid)
features_ref[:,1]=np.log(av[:,:5].sum(axis=1)+bv[:,:5].sum(axis=1))
features_ref[:,2]=(bv[:,0]-av[:,0])/(bv[:,0]+av[:,0])
features_ref[:,3]=(bv[:,1:5].sum(axis=1)-av[:,1:5].sum(axis=1))/(bv[:,1:5].sum(axis=1)+av[:,1:5].sum(axis=1))
i=np.flatnonzero(valid_prev); p=prev[i]
bid_comp=(b[i,0]>=b[p,0])*bv[i,0]-(b[i,0]<=b[p,0])*bv[p,0]
ask_comp=(a[i,0]>=a[p,0])*av[p,0]-(a[i,0]<=a[p,0])*av[i,0]
avg_depth=0.5*(bv[i,0]+av[i,0]+bv[p,0]+av[p,0])
features_ref[i,4]=(bid_comp+ask_comp)/avg_depth
features_ref[i,5]=np.log(mid[i]/mid[p])
for sid in np.unique(seq):
    ix=np.flatnonzero(seq==sid)
    features_ref[ix,6]=pd.Series(features_ref[ix,5]).rolling(20,min_periods=20).std(ddof=0).to_numpy()
max_feature_error={}
for k,name in enumerate(FEATURES+[REGIME]):
    observed=df[name].to_numpy(dtype=np.float64)
    expected=features_ref[:,k]
    if not np.array_equal(np.isnan(observed),np.isnan(expected)):
        raise RuntimeError(f'Feature missingness differs: {name}')
    delta=float(np.nanmax(np.abs(observed-expected)))
    max_feature_error[name]=delta
    if not np.allclose(observed, expected, rtol=1e-12,atol=1e-12,equal_nan=True):
        raise RuntimeError(f'Independent feature reconstruction mismatch: {name}; maximum error {delta}')

# Verify exact metadata, labels and end-of-sequence availability masks.
for name, expected in [('day',raw_day),('sample_in_day',sample),('sequence_id',seq),
                       ('instrument_segment',segments),('is_segment_start',starts)]:
    assert np.array_equal(df[name].to_numpy(),expected),name
for j,name in enumerate(['label_h10','label_h20','label_h30','label_h50','label_h100']):
    assert np.array_equal(df[name].to_numpy(),labels[:,j]),name
current_valid=np.isfinite(features_ref).all(axis=1)
assert current_valid.sum()==393337
assert np.allclose(Xt,features_ref[current_valid,:6],rtol=1e-12,atol=1e-12)
assert np.allclose(Xreg,features_ref[current_valid],rtol=1e-12,atol=1e-12)
assert np.array_equal(yt,labels[current_valid])
assert np.array_equal(dt,raw_day[current_valid]) and np.array_equal(st,seq[current_valid])
assert np.array_equal(segt,segments[current_valid]) and np.array_equal(pos,sample[current_valid])

independent_lag=np.zeros(len(raw),dtype=bool)
end_purge={h:0 for h in (10,20,30,50,100)}
for sid in np.unique(seq):
    ix=np.flatnonzero(seq==sid)
    assert len(ix)>120
    assert np.count_nonzero(~current_valid[ix])==20
    for p0 in range(25,len(ix)):
        independent_lag[ix[p0]]=current_valid[ix[p0-5:p0+1]].all()
    for h in end_purge:
        end_purge[h]+=min(h,len(ix))
assert independent_lag.sum()==393087
assert np.array_equal(old_lag_mask,independent_lag[current_valid])
assert end_purge=={10:500,20:1000,30:1500,50:2500,100:5000}

# Governance: only Days 1..5 become discovery data. D6–10 are not exposed to learner.
train_selector=np.isin(dt,DISCOVERY_DAYS)
assert np.count_nonzero(train_selector)>0 and np.all(dt[train_selector]<=5)
DATA_FINGERPRINT=sha_arrays(Xt[train_selector],dt[train_selector],st[train_selector])
preflight={'version':STAGE_VERSION, 'new_input_version':INPUT_VERSION,
           'boundary_audit':boundary_rows,'feature_max_abs_error':max_feature_error,
           'original_m1_exact_comparison':original_comparison,
           'full_lag_valid_rows':int(independent_lag.sum()),
           'train_days':list(DISCOVERY_DAYS),'validation_days':list(VALIDATION_DAYS),
           'historically_exposed_days':list(RESERVED_DAYS),
           'future_horizon_end_purge_counts':{str(k):v for k,v in end_purge.items()},
           'discovery_data_fingerprint':DATA_FINGERPRINT,
           'warning':'Publisher target labels not reconstructed yet; horizon-specific end purges required for downstream evaluation. D8-10 previously inspected, not external-blind.'}
atomic_json(RESULTS/'01_STEP2_INDEPENDENT_PREFLIGHT.json',preflight)
print('PREFLIGHT PASS: independent 7-feature reconstruction and all 10 boundary days.')
print('Old M1 exact checks:',sum(v is True for v in original_comparison.values()),
      'available; unverified:',sum(v=='OLD_CHECKPOINT_NOT_AVAILABLE' for v in original_comparison.values()))
print('Max absolute feature differences:',max_feature_error)
print('Lag-valid:',int(independent_lag.sum()),' | Discovery data hash:',DATA_FINGERPRINT)

# %%
# CELL 3 — Prepare ONLY D1–D5; sequence-wise scaling like the frozen historical M2
# Distinct sequences enter Tigramite as separate datasets; no pooled boundary crossings.
from collections import Counter

s0=np.flatnonzero(np.r_[True, st[1:]!=st[:-1]])
e0=np.r_[s0[1:],len(st)]
discovery_dict={}
norm_rows=[]
training_ids=[]
for j,(start,end) in enumerate(zip(s0,e0)):
    sid=int(st[start]); d=int(dt[start])
    assert np.all(st[start:end]==sid) and np.all(dt[start:end]==d)
    if d not in DISCOVERY_DAYS: continue
    arr=Xt[start:end]
    mu=arr.mean(axis=0)
    sd=arr.std(axis=0,ddof=0)
    if np.any(~np.isfinite(mu)) or np.any(sd<1e-12):
        raise RuntimeError(f'Non-finite/near-constant training feature in sequence {sid}')
    z=(arr-mu)/sd
    if not np.isfinite(z).all(): raise RuntimeError('Non-finite normalized training state')
    training_ids.append(sid)
    discovery_dict[len(discovery_dict)]=z
    for name,m,v in zip(FEATURES,mu,sd):
        norm_rows.append({'sequence_id':sid,'day':d,'feature':name,'mean':float(m),'std':float(v)})
assert len(discovery_dict)==25 and len(set(training_ids))==25
assert set(int(dt[np.flatnonzero(st==sid)[0]]) for sid in training_ids)==set(DISCOVERY_DAYS)
discovery_df=pp.DataFrame(data=discovery_dict,var_names=FEATURES,analysis_mode='multiple')
pd.DataFrame(norm_rows).to_csv(RESULTS/'02_D1_D5_discovery_scaling.csv',index=False)
print('Discovery sequences:',len(discovery_dict),' | Rows:',sum(len(a) for a in discovery_dict.values()))
print('Scaling: independently within each discovery sequence, using no D6–D10 state values.')

# %%
# CELL 4 — Frozen LPCMCI only; strict fingerprinted checkpoint, no stale M2 cache
LPCMCI_KWARGS=dict(
    tau_min=1,tau_max=5,pc_alpha=0.01,n_preliminary_iterations=1,
    max_p_global=5,max_p_non_ancestral=5,max_q_global=25,max_pds_set=15,
    prelim_only=False,no_non_ancestral_phase=False,break_once_separated=True,
    parents_of_lagged=True,fix_all_edges_before_final_orientation=True,
)
MODEL_SPEC={'algorithm':'LPCMCI','tigramite_version':DESIRED_TIGRAMITE,
            'test':'ParCorr','significance':'analytic',
            'kwargs':LPCMCI_KWARGS,'feature_names':FEATURES,
            'scale':'per-sequence z-score, ddof=0',
            'training_days':list(DISCOVERY_DAYS),'data_fingerprint':DATA_FINGERPRINT}
SPEC_HASH=hashlib.sha256(json.dumps(MODEL_SPEC,sort_keys=True).encode()).hexdigest()
PRIMARY=CKPT/'01_LPCMCI_fingerprinted.npz'
PRIMARY_META=CKPT/'01_LPCMCI_fingerprinted.json'

if PRIMARY.exists() and PRIMARY_META.exists() and not FORCE_DISCOVERY:
    previous=json.loads(PRIMARY_META.read_text(encoding='utf-8'))
    if previous.get('spec_hash')!=SPEC_HASH:
        raise RuntimeError('Existing discovery checkpoint is STALE: spec or data mismatch. '
                           'Preserve it and use a fresh output directory; do not silently load.')
    with np.load(PRIMARY,allow_pickle=False) as z:
        if str(z['spec_hash'].item())!=SPEC_HASH:
            raise RuntimeError('Checkpoint file/manifest fingerprint disagreement.')
        graph=z['graph']; p_matrix=z['p_matrix']; val_matrix=z['val_matrix']
    print('Reused EXACT-SPEC discovery checkpoint:',PRIMARY)
else:
    if PRIMARY.exists() and not FORCE_DISCOVERY and not PRIMARY_META.exists():
        raise RuntimeError('Checkpoint exists without manifest; manual review required.')
    print('Running frozen LPCMCI on D1–D5; this can take a while...')
    begin=time.time()
    learner=LPCMCI(dataframe=discovery_df,
                   cond_ind_test=ParCorr(significance='analytic'),verbosity=0)
    result=learner.run_lpcmci(**LPCMCI_KWARGS)
    graph=result['graph']; p_matrix=result['p_matrix']; val_matrix=result['val_matrix']
    runtime=time.time()-begin
    tmp=PRIMARY.with_name(PRIMARY.name+'.tmp.npz')
    np.savez_compressed(tmp,graph=graph,p_matrix=p_matrix,val_matrix=val_matrix,
                        feature_names=np.asarray(FEATURES),spec_hash=np.asarray(SPEC_HASH))
    tmp.replace(PRIMARY)
    atomic_json(PRIMARY_META,{'spec_hash':SPEC_HASH,'model_spec':MODEL_SPEC,
                              'runtime_sec':runtime,'checkpoint':str(PRIMARY)})
    print(f'Primary discovery saved in {runtime:.1f}s.')

assert graph.shape==(6,6,6),f'Unexpected graph shape {graph.shape}'
assert p_matrix.shape==graph.shape and val_matrix.shape==graph.shape
assert not np.any(graph[:,:,0]!=''),'Lag-zero edges violate frozen spec.'
assert np.isfinite(val_matrix[graph!='']).all()
print('Primary discovery fingerprint:',SPEC_HASH)

# %%
# CELL 5 — Parse all edge marks conservatively; export candidate families, no selection yet
records=[]
for lag in range(1,LAGS+1):
    for source in range(6):
        for target in range(6):
            mark=str(graph[source,target,lag])
            if not mark: continue
            orientation=('FULLY_DIRECTED_MARK' if mark=='-->'
                         else 'AMBIGUOUS_OR_OTHER_MARK')
            records.append({'source':FEATURES[source],'target':FEATURES[target],
                            'lag':lag,'edge_mark':mark,'orientation_class':orientation,
                            'self_edge':bool(source==target),
                            'parcorr_stat':float(val_matrix[source,target,lag]),
                            'algorithmic_p_value':float(p_matrix[source,target,lag])})

cols=['source','target','lag','edge_mark','orientation_class','self_edge',
      'parcorr_stat','algorithmic_p_value']
edges=pd.DataFrame(records,columns=cols)
edges.to_csv(RESULTS/'03_LPCMCI_ALL_MARKS.csv',index=False)

# Family table is descriptive only. Robustness and edge selection occur later on D6–D7.
cross=edges.loc[~edges.self_edge].copy()
family=[]
for (s,t),group in cross.groupby(['source','target'],sort=True):
    family.append({'source':s,'target':t,
                   'lags':','.join(map(str,sorted(group.lag.astype(int).unique()))),
                   'edge_marks':','.join(sorted(group.edge_mark.unique())),
                   'has_nonfully_directed_mark':bool((group.orientation_class!='FULLY_DIRECTED_MARK').any()),
                   'n_lag_slots':int(len(group))})
families=pd.DataFrame(family,columns=['source','target','lags','edge_marks',
                                    'has_nonfully_directed_mark','n_lag_slots'])
families.to_csv(RESULTS/'04_DESCRIPTIVE_CROSS_FAMILIES.csv',index=False)
mark_count={str(k):int(v) for k,v in Counter(edges.edge_mark).items()}
summary={'version':STAGE_VERSION,'status':'DISCOVERY_ONLY_NOT_VALIDATED',
         'training_days':[1,2,3,4,5],'validation_days_not_used':[6,7],
         'previously_exposed_days_not_used':[8,9,10],
         'fingerprint':SPEC_HASH,'retained_lag_slots':int(len(edges)),
         'self_slots':int(edges.self_edge.sum()),
         'cross_slots':int((~edges.self_edge).sum()),
         'candidate_cross_families':int(len(families)),
         'mark_counts':mark_count,
         'caveat':'Graph and family CSVs are candidates only. Do not call these identified causal effects or frozen M3 edges.'}
atomic_json(RESULTS/'05_DISCOVERY_ONLY_SUMMARY.json',summary)
print('STEP 3 DISCOVERY COMPLETE (not validated / not graph-frozen):')
print(json.dumps(summary,indent=2))
print('\nDiscovered candidate families:')
print(families.to_string(index=False))
print('\nNext step: D6–D7 prespecified family falsification and PCMCI+ sensitivity; do NOT access D8–D10 for graph selection.')

Google Drive already mounted.
Input (read-only): /content/drive/MyDrive/CausalLOB/STEP2_REBUILD/checkpoints
New output root  : /content/drive/MyDrive/CausalLOB/STEP3_REBUILD_FROZEN
Discovery D1–5; internal validation D6–7; historically exposed D8–10.
PREFLIGHT PASS: independent 7-feature reconstruction and all 10 boundary days.
Old M1 exact checks: 14 available; unverified: 0
Max absolute feature differences: {'log_relative_spread': 0.0, 'log_total_depth_l5': 0.0, 'imbalance_l1': 0.0, 'imbalance_l2_l5': 0.0, 'quote_ofi_depth_norm': 0.0, 'log_mid_return_1': 0.0, 'event_volatility_20': 0.0}
Lag-valid: 393087  | Discovery data hash: 674705f93839244b4f06917736a3b383c3a95d794d10f9ff836c9d16a559b0e9
Discovery sequences: 25  | Rows: 177752
Scaling: independently within each discovery sequence, using no D6–D10 state values.
Running frozen LPCMCI on D1–D5; this can take a while...
Primary discovery saved in 1590.2s.
Primary discovery fingerprint: fa32b947d8ece5ff253e8784eb0ae2cfa58e068a9951841e

## D6-D7 structure validation and graph freeze

In [ ]:
# %% [markdown]
# # CausalLOB — Step 4: D6–D7 validation and graph freeze
#
# the NEW, fingerprinted Step 3 discovery output. Do not rerun LPCMCI. Independently check its graph
# and D1–D5 fingerprint; never use D8–D10 state or target data to select edges. Test each directed
# source→target family against an equal-history AR(5) Ridge baseline on D6 and D7, an independently
# specified nonlinear model, an explicitly anti-causal lead placebo, and a 100-replicate
# within-sequence phase-alignment null. Apply BH correction across ALL directed cross-families.
# Run PCMCI+ sensitivity on D1–D5 ONLY. Audit all 150 cross-variable lag slots with matching rows
# and a target×lag-matched random-set analysis (descriptive, not causal significance).
# Freeze AR(5) self-history plus ONLY families that satisfy the historical ROBUST rule; a null
# result or empty graph is permitted. Checkpoint stages to prevent expensive reruns.
#
# The graph is observational, not an identified causal model. A phase null is not an exchangeable
# statistical randomization test; the lead placebo is deliberately non-deployable. The D6–D7
# evidence has been inspected in previous CausalLOB research, so this is a reproducibility audit,
# NOT a newly prospective independent validation. D8–D10 are historically exposed too.
#
# Run every cell in sequence in Google Colab (CPU). Source and discovery checkpoints are READ-ONLY.
# Output: /content/drive/MyDrive/CausalLOB/STEP4_VALIDATION_FROZEN

# %%
# CELL 1 — exact provenance, chronology isolation and immutable audit configuration
import os, sys, subprocess, importlib.metadata, importlib.util
from pathlib import Path
from collections import Counter
import json, hashlib, time
import numpy as np
import pandas as pd
import sklearn
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor

try:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive')
    else:
        print('Google Drive already mounted.')
except ImportError:
    pass

ROOT=Path(os.environ.get('CAUSALLOB_ROOT','/content/drive/MyDrive/CausalLOB'))
STEP2=ROOT/'STEP2_REBUILD'/'checkpoints'
STEP3=ROOT/'STEP3_REBUILD_FROZEN'
IN_CKPT=STEP3/'checkpoints'
IN_RESULTS=STEP3/'results'
OUT=ROOT/'STEP4_VALIDATION_FROZEN'
CKPT=OUT/'checkpoints'
RESULTS=OUT/'results'
FAMILY_CACHE=CKPT/'family_jobs'
PCMCI_CACHE=CKPT/'pcmci_jobs'
for folder in (CKPT,RESULTS,FAMILY_CACHE,PCMCI_CACHE):
    folder.mkdir(parents=True,exist_ok=True)

FEATURES=['log_relative_spread','log_total_depth_l5','imbalance_l1',
          'imbalance_l2_l5','quote_ofi_depth_norm','log_mid_return_1']
IDX={x:i for i,x in enumerate(FEATURES)}
TRAIN_DAYS=(1,2,3,4,5)
VAL_DAYS=(6,7)
EXPOSED_DAYS=(8,9,10)
MAX_LAG=5
RIDGE_ALPHA=1.0
HGB_KWARGS=dict(loss='squared_error',learning_rate=0.05,max_iter=120,
                max_leaf_nodes=15,min_samples_leaf=50,l2_regularization=1.0,
                early_stopping=False,random_state=5162026)
PHASE_REPS=100
PHASE_SEED=5162026
PCMCI_TAUS=(1,3,5)
PCMCI_VERSION='5.2.10.1'
STAGE_VERSION='STEP4_VALIDATION_v1_2026-09-18'
PERMUTATIONS=5000
T0=time.time()

# Never add a FORCE flag: a mismatched output directory is a HARD FAILURE.
def sha_arrays(*arrays):
    h=hashlib.sha256()
    for a in arrays:
        a=np.ascontiguousarray(a)
        h.update(str(a.shape).encode());h.update(str(a.dtype).encode())
        h.update(a.view(np.uint8))
    return h.hexdigest()

def sha_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(1024*1024),b''):
            h.update(chunk)
    return h.hexdigest()

def atomic_json(path,obj):
    path=Path(path);tmp=path.with_suffix(path.suffix+'.tmp')
    tmp.write_text(json.dumps(obj,indent=2,allow_nan=False),encoding='utf-8')
    tmp.replace(path)

def atomic_csv(frame,path):
    path=Path(path);tmp=path.with_suffix(path.suffix+'.tmp')
    frame.to_csv(tmp,index=False);tmp.replace(path)

def atomic_npz(path,**arrays):
    path=Path(path);tmp=path.with_name(path.name+'.tmp.npz')
    np.savez_compressed(tmp,**arrays);tmp.replace(path)

def require_file(path):
    if not path.is_file():
        raise FileNotFoundError('Required immutable input missing: '+str(path))
    return path

raw_npz=require_file(STEP2/'04_temporal_primary.npz')
preflight=require_file(IN_RESULTS/'01_STEP2_INDEPENDENT_PREFLIGHT.json')
discovery_meta=require_file(IN_CKPT/'01_LPCMCI_fingerprinted.json')
discovery_npz=require_file(IN_CKPT/'01_LPCMCI_fingerprinted.npz')
discovery_summary=require_file(IN_RESULTS/'05_DISCOVERY_ONLY_SUMMARY.json')
scaling_csv=require_file(IN_RESULTS/'02_D1_D5_discovery_scaling.csv')
old_pre=json.loads(preflight.read_text())
meta=json.loads(discovery_meta.read_text())
sum3=json.loads(discovery_summary.read_text())
assert old_pre['version']=='STEP3_DISCOVERY_v1_2026-09-17'
assert sum3['status']=='DISCOVERY_ONLY_NOT_VALIDATED'
assert sum3['training_days']==list(TRAIN_DAYS)
assert sum3['validation_days_not_used']==list(VAL_DAYS)
assert sum3['previously_exposed_days_not_used']==list(EXPOSED_DAYS)
assert old_pre['train_days']==list(TRAIN_DAYS)
assert old_pre['validation_days']==list(VAL_DAYS)
assert all(row['exact_cut_match'] and row['fourth_to_fifth_jump_ratio']>10
           for row in old_pre['boundary_audit'])
assert sum(v is False for v in old_pre['original_m1_exact_comparison'].values())==0
assert all(v==0.0 for v in old_pre['feature_max_abs_error'].values())
assert sum3['retained_lag_slots']==59 and sum3['mark_counts']=={'-->':54,'<->':5}

# The NPZ holds all ten days, but isolate D1–D7 immediately; labels are NEVER loaded.
with np.load(raw_npz,allow_pickle=False) as z:
    assert str(z['checkpoint_version'].item())=='STEP2_REBUILD_v1_2026-09-17'
    assert z['feature_names'].astype(str).tolist()==FEATURES
    all_day=z['day'];all_seq=z['sequence_id']
    assert np.array_equal(np.unique(all_day),np.arange(1,11))
    use=all_day<=7
    X=z['X_current'][use].copy()
    day=all_day[use].copy()
    seq=all_seq[use].copy()
    del all_day,all_seq,use
assert X.ndim==2 and X.shape[1]==6 and np.isfinite(X).all()
assert np.array_equal(np.unique(day),np.arange(1,8))
assert len(np.unique(seq))==35
assert np.all(np.diff(day)>=0)
DISCOVERY_HASH=sha_arrays(X[day<=5],day[day<=5],seq[day<=5])
assert DISCOVERY_HASH==old_pre['discovery_data_fingerprint'], 'D1–D5 state differs from Stage 3!'
assert DISCOVERY_HASH==meta['model_spec']['data_fingerprint']
assert meta['model_spec']['training_days']==list(TRAIN_DAYS)
assert meta['model_spec']['feature_names']==FEATURES
assert meta['model_spec']['scale']=='per-sequence z-score, ddof=0'
assert meta['model_spec']['tigramite_version']==PCMCI_VERSION
assert meta['model_spec']['test']=='ParCorr'
assert meta['model_spec']['kwargs']['tau_min']==1
assert meta['model_spec']['kwargs']['tau_max']==5
assert meta['model_spec']['kwargs']['pc_alpha']==0.01
assert hashlib.sha256(json.dumps(meta['model_spec'],sort_keys=True).encode()).hexdigest()==meta['spec_hash']
assert meta['spec_hash']==sum3['fingerprint']

with np.load(discovery_npz,allow_pickle=False) as z:
    assert str(z['spec_hash'].item())==meta['spec_hash']
    assert z['feature_names'].astype(str).tolist()==FEATURES
    graph=z['graph'].copy();pvals=z['p_matrix'].copy();statistics=z['val_matrix'].copy()
assert graph.shape==(6,6,6) and pvals.shape==graph.shape and statistics.shape==graph.shape
assert not np.any(graph[:,:,0]!='')
assert set(graph.flatten())<= {'','-->','<->'}
assert int(np.count_nonzero(graph!=''))==59
assert int(np.count_nonzero(graph=='-->'))==54
assert int(np.count_nonzero(graph=='<->'))==5

edges=[]
for lag in range(1,6):
    for source in range(6):
        for target in range(6):
            mark=str(graph[source,target,lag])
            if mark:
                edges.append({'source':FEATURES[source],'target':FEATURES[target],
                              'lag':lag,'edge_mark':mark,'self_edge':source==target,
                              'parcorr_stat':float(statistics[source,target,lag])})
all_edges=pd.DataFrame(edges)
assert len(all_edges)==59
assert sum((all_edges.edge_mark=='-->') & (~all_edges.self_edge))==27
# Independently verify that the saved Step 3 tabular graph matches the actual checkpoint.
step3_csv=require_file(IN_RESULTS/'03_LPCMCI_ALL_MARKS.csv')
stored=pd.read_csv(step3_csv)
key=lambda d:set(zip(d.source.astype(str),d.target.astype(str),d.lag.astype(int),d.edge_mark.astype(str)))
assert key(all_edges)==key(stored),'Step 3 CSV/NPZ disagreement'

# Historical v2 only validates FULLY DIRECTED cross families; do not silently orient ambiguous edges.
directed_cross=all_edges[(all_edges.edge_mark=='-->') & (~all_edges.self_edge)].copy()
ambiguous=all_edges[all_edges.edge_mark!='-->'].copy()
family_info=[]
for (source,target),grp in directed_cross.groupby(['source','target'],sort=True):
    reciprocal=((directed_cross.source==target)&(directed_cross.target==source)).any()
    ambiguous_pair=((ambiguous.source==source)&(ambiguous.target==target) |
                    (ambiguous.source==target)&(ambiguous.target==source)).any()
    family_info.append({'source':source,'target':target,
                        'discovery_lags':','.join(map(str,sorted(grp.lag.astype(int).unique()))),
                        'reciprocal_directed_family':bool(reciprocal),
                        'pair_has_ambiguous_endpoint':bool(ambiguous_pair)})
families=pd.DataFrame(family_info)
assert len(families)==16,f'Expected 16 directed families, got {len(families)}'

# Validate history slices and stored D1–D5 normalization. No scaling from D6–D7 is used for training.
starts=np.flatnonzero(np.r_[True,seq[1:]!=seq[:-1]])
ends=np.r_[starts[1:],len(seq)]
SLICES={}
for a,b in zip(starts,ends):
    sid=int(seq[a]);d=int(day[a]);assert b-a>20
    assert np.all(seq[a:b]==sid) and np.all(day[a:b]==d)
    SLICES[sid]=(slice(int(a),int(b)),d)
assert len(SLICES)==35
saved_scaling=pd.read_csv(scaling_csv)
assert len(saved_scaling)==150
for sid,(sl,d) in SLICES.items():
    if d>5:continue
    arr=X[sl];mu=arr.mean(axis=0);sd=arr.std(axis=0,ddof=0)
    row=saved_scaling[saved_scaling.sequence_id==sid].set_index('feature').loc[FEATURES]
    assert np.allclose(row['mean'].to_numpy(),mu,rtol=1e-12,atol=1e-12)
    assert np.allclose(row['std'].to_numpy(),sd,rtol=1e-12,atol=1e-12)

VALIDATION_HASH=sha_arrays(X[day>=6],day[day>=6],seq[day>=6])
RUN_SPEC={'stage':STAGE_VERSION,'discovery_spec_sha256':meta['spec_hash'],
          'discovery_checkpoint_sha256':sha_file(discovery_npz),
          'train_data_sha256':DISCOVERY_HASH,'validation_data_sha256':VALIDATION_HASH,
          'features':FEATURES,'days_train':list(TRAIN_DAYS),'days_validation':list(VAL_DAYS),
          'ridge_alpha':RIDGE_ALPHA,'hgb_kwargs':HGB_KWARGS,
          'phase_reps':PHASE_REPS,'phase_seed':PHASE_SEED,
          'pcmci_taus':list(PCMCI_TAUS),'pcmci_version':PCMCI_VERSION,
          'permutations':PERMUTATIONS,'sklearn_version':sklearn.__version__,
          'method':'historical M2 v2 robustness gates; all-slot matched-row diagnostic'}
RUN_HASH=hashlib.sha256(json.dumps(RUN_SPEC,sort_keys=True).encode()).hexdigest()
run_path=CKPT/'IMMUTABLE_RUN_SPEC.json'
if run_path.exists():
    prior=json.loads(run_path.read_text())
    if prior['run_hash']!=RUN_HASH:
        raise RuntimeError('Existing Step 4 outputs have a DIFFERENT data/spec/version fingerprint. '
                           'Preserve them and use a NEW output directory; NEVER mix caches.')
else:
    # If an incomplete/corrupt run left caches but no manifest, do not accept unversioned artifacts.
    if any(FAMILY_CACHE.iterdir()) or any(PCMCI_CACHE.iterdir()):
        raise RuntimeError('Output contains work but no provenance manifest. Manual inspection required.')
    atomic_json(run_path,{'run_hash':RUN_HASH,'spec':RUN_SPEC})

atomic_csv(all_edges,RESULTS/'01_DISCOVERY_GRAPH_VERIFIED.csv')
atomic_csv(families,RESULTS/'02_DIRECTED_CROSS_FAMILIES.csv')
print('STEP 4 PREFLIGHT PASS; no D8–D10 state or label used.')
print('Step 3 discovery fingerprint:',meta['spec_hash'])
print('Step 4 run fingerprint:',RUN_HASH)
print('59 candidate links / 27 directed cross slots / 16 directed families / 5 ambiguous marks.')
print('Train D1–5:',int((day<=5).sum()),'Validation D6–7:',int((day>=6).sum()))
print('Outputs:',OUT)

# %%
# CELL 2 — frozen family-testing helpers; equality-of-rows guards

def family_dataset(source,target,lags,days,mode='lag',fixed_end=None):
    """Paired AR5 and AR5+family matrices; same rows, same own-history, no crossings."""
    lags=tuple(sorted(set(map(int,lags))))
    assert lags and set(lags)<=set(range(1,6))
    assert source!=target and source in IDX and target in IDX
    assert mode in ('lag','lead')
    mx=max(lags) if fixed_end is None else int(fixed_end)
    assert mx>=max(lags) and mx<=5
    si=IDX[source];ti=IDX[target]
    bx=[];ax=[];yy=[];dd=[];ss=[]
    for sid,(sl,d) in SLICES.items():
        if d not in days:continue
        arr=X[sl]
        pos=np.arange(5,len(arr)-mx,dtype=np.int64)
        if not len(pos):continue
        baseline=np.column_stack([arr[pos-l,ti] for l in range(1,6)])
        suffix=np.column_stack([arr[pos-l if mode=='lag' else pos+l,si] for l in lags])
        bx.append(baseline);ax.append(np.column_stack([baseline,suffix]))
        yy.append(arr[pos,ti]);dd.append(np.full(len(pos),d,dtype=np.int8))
        ss.append(np.full(len(pos),sid,dtype=np.int16))
    if not bx:raise RuntimeError('No eligible rows')
    B=np.vstack(bx);A=np.vstack(ax);y=np.concatenate(yy)
    days_arr=np.concatenate(dd);seq_arr=np.concatenate(ss)
    assert A.shape[0]==B.shape[0]==len(y)==len(days_arr)==len(seq_arr)
    assert A.shape[1]==5+len(lags) and np.array_equal(A[:,:5],B)
    assert np.isfinite(A).all() and np.isfinite(B).all() and np.isfinite(y).all()
    return B,A,y,days_arr,seq_arr

def paired_ridge(train_b,train_a,train_y):
    mu=float(train_y.mean());sd=float(train_y.std(ddof=0))
    assert np.isfinite(mu) and np.isfinite(sd) and sd>1e-12
    z=(train_y-mu)/sd
    baseline=make_pipeline(StandardScaler(),Ridge(alpha=RIDGE_ALPHA))
    augmented=make_pipeline(StandardScaler(),Ridge(alpha=RIDGE_ALPHA))
    baseline.fit(train_b,z);augmented.fit(train_a,z)
    return baseline,augmented,mu,sd

def prediction_deltas(b,a,y,days,model_b,model_a,mu,sd):
    yz=(y-mu)/sd
    pb=model_b.predict(b);pa=model_a.predict(a)
    out={}
    for d in VAL_DAYS:
        ix=(days==d)
        assert ix.sum()>0
        out[d]={'n':int(ix.sum()),'delta_mse':float(np.mean((yz[ix]-pb[ix])**2)-
                                                      np.mean((yz[ix]-pa[ix])**2))}
    pooled=float(np.mean((yz-pb)**2)-np.mean((yz-pa)**2))
    assert abs(pooled-np.average([out[d]['delta_mse'] for d in VAL_DAYS],
                                 weights=[out[d]['n'] for d in VAL_DAYS]))<1e-10
    return out,pooled

def family_seed(source,target):
    raw=f'{PHASE_SEED}|{RUN_HASH}|{source}|{target}'.encode()
    return int.from_bytes(hashlib.sha256(raw).digest()[:8],'big')

def phase_null(source,target,lags,mb,ma,mu,sd,B,A,y,days,seq_ids):
    yz=(y-mu)/sd
    base_pred=mb.predict(B)
    actual_pred=ma.predict(A)
    base_mse=float(np.mean((yz-base_pred)**2))
    actual_gain=base_mse-float(np.mean((yz-actual_pred)**2))
    pos_by_seq=[np.flatnonzero(seq_ids==sid) for sid in np.unique(seq_ids)]
    rng=np.random.default_rng(family_seed(source,target))
    source_block=A[:,5:].copy()
    null_gains=np.empty(PHASE_REPS,dtype=np.float64)
    for j in range(PHASE_REPS):
        perturbed=A.copy()
        for indices in pos_by_seq:
            n=len(indices)
            low=max(10,2*max(lags)+1)
            high=n-low
            if high<=low:
                raise RuntimeError('Validation sequence too short for controlled phase shift')
            k=int(rng.integers(low,high))
            perturbed[indices,5:]=np.roll(source_block[indices],k,axis=0)
        null_pred=ma.predict(perturbed)
        null_gains[j]=base_mse-float(np.mean((yz-null_pred)**2))
    p=(1+int(np.count_nonzero(null_gains>=actual_gain)))/(PHASE_REPS+1)
    return dict(phase_empirical_p=float(p),phase_null_mean_gain=float(null_gains.mean()),
                phase_null_q95_gain=float(np.quantile(null_gains,0.95)),
                phase_actual_gain=float(actual_gain),phase_null_reps=PHASE_REPS)

def bh_qvalues(pvals):
    p=np.asarray(pvals,dtype=np.float64)
    assert p.ndim==1 and len(p)>0 and np.isfinite(p).all() and np.all((p>=0)&(p<=1))
    order=np.argsort(p)
    tmp=p[order]*len(p)/np.arange(1,len(p)+1)
    tmp=np.minimum.accumulate(tmp[::-1])[::-1]
    out=np.empty(len(p));out[order]=np.clip(tmp,0,1)
    return out

def evaluate_one_family(info):
    source=str(info.source);target=str(info.target)
    lags=tuple(map(int,str(info.discovery_lags).split(',')))
    Btr,Atr,ytr,dtr,str_=family_dataset(source,target,lags,TRAIN_DAYS)
    Bva,Ava,yva,dva,sva=family_dataset(source,target,lags,VAL_DAYS)
    assert set(np.unique(dtr))==set(TRAIN_DAYS)
    assert set(np.unique(dva))==set(VAL_DAYS)
    mb,ma,mu,sd=paired_ridge(Btr,Atr,ytr)
    ridge,delta=prediction_deltas(Bva,Ava,yva,dva,mb,ma,mu,sd)
    ztr=(ytr-mu)/sd
    hgb_b=HistGradientBoostingRegressor(**HGB_KWARGS)
    hgb_a=HistGradientBoostingRegressor(**HGB_KWARGS)
    hgb_b.fit(Btr,ztr);hgb_a.fit(Atr,ztr)
    hgb,hgb_delta=prediction_deltas(Bva,Ava,yva,dva,hgb_b,hgb_a,mu,sd)
    # Explicit anti-causal negative control. Do NOT use lead observations in any forecasting model.
    Btlead,Atlead,ytlead,dtlead,_=family_dataset(source,target,lags,TRAIN_DAYS,'lead')
    Bvlead,Avlead,yvlead,dvlead,_=family_dataset(source,target,lags,VAL_DAYS,'lead')
    assert np.array_equal(Btr,Btlead) and np.array_equal(Bva,Bvlead)
    assert np.array_equal(ytr,ytlead) and np.array_equal(yva,yvlead)
    lead=make_pipeline(StandardScaler(),Ridge(alpha=RIDGE_ALPHA))
    lead.fit(Atlead,ztr)
    lead_day,lead_delta=prediction_deltas(Bvlead,Avlead,yvlead,dvlead,mb,lead,mu,sd)
    null=phase_null(source,target,lags,mb,ma,mu,sd,Bva,Ava,yva,dva,sva)
    assert abs(null['phase_actual_gain']-delta)<1e-10
    return {'source':source,'target':target,'discovery_lags':info.discovery_lags,
            'n_train':int(len(ytr)),'n_D6':int(ridge[6]['n']),'n_D7':int(ridge[7]['n']),
            'ridge_D6_delta_mse':ridge[6]['delta_mse'],
            'ridge_D7_delta_mse':ridge[7]['delta_mse'],
            'ridge_positive_days':int(sum(ridge[d]['delta_mse']>0 for d in VAL_DAYS)),
            'ridge_weighted_delta_mse':delta,
            'hgb_D6_delta_mse':hgb[6]['delta_mse'],
            'hgb_D7_delta_mse':hgb[7]['delta_mse'],
            'hgb_weighted_delta_mse':hgb_delta,
            'hgb_positive_days':int(sum(hgb[d]['delta_mse']>0 for d in VAL_DAYS)),
            'lead_placebo_weighted_delta_mse':lead_delta,
            'lag_minus_lead_gain':delta-lead_delta,
            'reciprocal_directed_family':bool(info.reciprocal_directed_family),
            'pair_has_ambiguous_endpoint':bool(info.pair_has_ambiguous_endpoint),
            **null}

# %%
# CELL 3 — D6–D7 family falsification; durable checkpoint per family
family_rows=[]
for number,info in enumerate(families.itertuples(index=False),start=1):
    target_file=FAMILY_CACHE/f'family_{number:02d}.json'
    if target_file.exists():
        job=json.loads(target_file.read_text())
        if job.get('run_hash')!=RUN_HASH or job.get('source')!=info.source or job.get('target')!=info.target:
            raise RuntimeError('Family job checkpoint fingerprint/identity mismatch: '+str(target_file))
        row=job['result']
        print(f'Cached family {number:02d}/{len(families)}: {info.source} -> {info.target}')
    else:
        started=time.time()
        row=evaluate_one_family(info)
        atomic_json(target_file,{'run_hash':RUN_HASH,'source':info.source,
                                 'target':info.target,'result':row,
                                 'runtime_sec':time.time()-started})
        print(f'Finished family {number:02d}/{len(families)}: {info.source} -> {info.target}; '
              f'Ridge ΔMSE={row["ridge_weighted_delta_mse"]:+.6g}; '
              f'phase p={row["phase_empirical_p"]:.4f}')
    family_rows.append(row)

family_df=pd.DataFrame(family_rows)
assert len(family_df)==16 and not family_df[['ridge_weighted_delta_mse',
                                           'hgb_weighted_delta_mse','phase_empirical_p']].isna().any().any()
family_df['phase_bh_q']=bh_qvalues(family_df.phase_empirical_p.to_numpy())
atomic_csv(family_df,RESULTS/'03_ALL_FAMILY_FALSIFICATION.csv')
print('\nD6–D7 directed-family falsification complete; p/q are descriptive alignment diagnostics.')
print(family_df[['source','target','discovery_lags','ridge_D6_delta_mse',
                 'ridge_D7_delta_mse','hgb_weighted_delta_mse','lag_minus_lead_gain',
                 'phase_empirical_p','phase_bh_q']].to_string(index=False))

# %%
# CELL 4 — train-only PCMCI+ [tau=1,3,5] model-family sensitivity
try:
    installed=importlib.metadata.version('tigramite')
except importlib.metadata.PackageNotFoundError:
    installed=None
if installed!=PCMCI_VERSION:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',f'tigramite=={PCMCI_VERSION}'])
from tigramite import data_processing as pp
from tigramite.pcmci import PCMCI
from tigramite.independence_tests.parcorr import ParCorr

normal_data={}
for sid,(sl,d) in SLICES.items():
    if d not in TRAIN_DAYS:continue
    arr=X[sl]
    norm_data=(arr-arr.mean(axis=0))/arr.std(axis=0,ddof=0)
    assert np.isfinite(norm_data).all()
    normal_data[len(normal_data)]=norm_data
assert len(normal_data)==25
train_frame=pp.DataFrame(data=normal_data,var_names=FEATURES,analysis_mode='multiple')
PCMCI_KWARGS=dict(tau_min=1,pc_alpha=0.01,contemp_collider_rule='majority',
                  conflict_resolution=True,reset_lagged_links=False,max_conds_dim=5,
                  max_combinations=1,max_conds_py=5,max_conds_px=5,
                  max_conds_px_lagged=5,fdr_method='fdr_bh')

pcmci_sets={}
pcmi_rows=[]
for tau in PCMCI_TAUS:
    spec={'step4_run_hash':RUN_HASH,'algorithm':'PCMCI+',
          'tigramite':PCMCI_VERSION,'test':'ParCorr analytic',
          'tau_max':tau,'kwargs':PCMCI_KWARGS,'days':[1,2,3,4,5]}
    hash_pcmci=hashlib.sha256(json.dumps(spec,sort_keys=True).encode()).hexdigest()
    path=PCMCI_CACHE/f'pcmci_tau{tau}.npz'
    manifest=PCMCI_CACHE/f'pcmci_tau{tau}.json'
    if path.exists() or manifest.exists():
        if not (path.exists() and manifest.exists()):
            raise RuntimeError(f'Incomplete PCMCI+ tau={tau} cache; manual inspection required')
        j=json.loads(manifest.read_text())
        assert j['spec_hash']==hash_pcmci
        with np.load(path,allow_pickle=False) as z:
            assert str(z['spec_hash'].item())==hash_pcmci
            pg=z['graph'].copy();pv=z['p_matrix'].copy();vv=z['val_matrix'].copy()
        print(f'Cached D1–D5 PCMCI+ tau={tau}.')
    else:
        print(f'Running D1–D5 PCMCI+ tau={tau}; no D6–D10 used...')
        start=time.time()
        learner=PCMCI(dataframe=train_frame,cond_ind_test=ParCorr(significance='analytic'),verbosity=0)
        result=learner.run_pcmciplus(tau_max=tau,**PCMCI_KWARGS)
        pg=result['graph'];pv=result['p_matrix'];vv=result['val_matrix']
        assert pg.shape==(6,6,tau+1)
        atomic_npz(path,graph=pg,p_matrix=pv,val_matrix=vv,spec_hash=np.array(hash_pcmci))
        atomic_json(manifest,{'spec_hash':hash_pcmci,'spec':spec,
                              'runtime_sec':time.time()-start})
    assert pg.shape==(6,6,tau+1) and pv.shape==pg.shape and vv.shape==pg.shape
    lagged={(FEATURES[s],FEATURES[t],int(k))
            for k in range(1,tau+1) for s in range(6) for t in range(6)
            if str(pg[s,t,k]) not in ('','<--')}
    pcmci_sets[tau]=lagged
    eligible={(str(r.source),str(r.target),int(r.lag))
              for r in directed_cross.itertuples() if int(r.lag)<=tau}
    common=eligible & lagged
    union=eligible | lagged
    pcmi_rows.append({'tau_max':tau,'pcmci_lagged_slots':len(lagged),
                      'eligible_lpcmci_directed_slots':len(eligible),
                      'common_slots':len(common),
                      'lpcmci_coverage':len(common)/len(eligible) if eligible else None,
                      'jaccard':len(common)/len(union) if union else None})
pcmi_df=pd.DataFrame(pcmi_rows)
atomic_csv(pcmi_df,RESULTS/'04_PCMCIplus_tau_sensitivity.csv')

supports=[]
for r in families.itertuples(index=False):
    lags=tuple(map(int,r.discovery_lags.split(',')))
    by_lag=[]
    for lag in lags:
        eligible_tau=[t for t in PCMCI_TAUS if t>=lag]
        if eligible_tau:
            by_lag.append(np.mean([(r.source,r.target,lag) in pcmci_sets[t]
                                   for t in eligible_tau]))
    supports.append({'source':r.source,'target':r.target,
                     'pcmci_family_mean_support':float(np.mean(by_lag)),
                     'pcmci_family_max_support':float(np.max(by_lag))})
support_df=pd.DataFrame(supports)
atomic_csv(support_df,RESULTS/'05_PCMCIplus_family_support.csv')
print('\nPCMCI+ sensitivity complete — model-family agreement is descriptive, not causal proof.')
print(pcmi_df.to_string(index=False))

# %%
# CELL 5 — all 150 cross-slots, equal-row comparisons, target×lag-matched set sensitivity
slots_file=CKPT/'150_slots_fingerprinted.csv'
slots_meta=CKPT/'150_slots_fingerprinted.json'
if slots_file.exists() or slots_meta.exists():
    if not (slots_file.exists() and slots_meta.exists()):
        raise RuntimeError('Incomplete all-slot checkpoint; inspect before continuing')
    audit=json.loads(slots_meta.read_text())
    if audit['run_hash']!=RUN_HASH or audit['sha256']!=sha_file(slots_file):
        raise RuntimeError('All-slot checkpoint hash mismatch')
    slots=pd.read_csv(slots_file)
    print('Reused fingerprinted 150-slot checkpoint.')
else:
    rows=[]
    for target in FEATURES:
        for source in FEATURES:
            if source==target:continue
            for lag in range(1,6):
                # fixed_end=5 for EVERY candidate, so lag comparisons use matched rows.
                Bt,At,yt,dt_,st_=family_dataset(source,target,[lag],TRAIN_DAYS,fixed_end=5)
                Bv,Av,yv,dv,sv=family_dataset(source,target,[lag],VAL_DAYS,fixed_end=5)
                mb,ma,mu,sd=paired_ridge(Bt,At,yt)
                by_day,gain=prediction_deltas(Bv,Av,yv,dv,mb,ma,mu,sd)
                rows.append({'source':source,'target':target,'lag':lag,
                             'n_D6':by_day[6]['n'],'n_D7':by_day[7]['n'],
                             'D6_delta_mse':by_day[6]['delta_mse'],
                             'D7_delta_mse':by_day[7]['delta_mse'],
                             'weighted_delta_mse':gain})
        print(f'All-slot evaluation finished target: {target}')
    slots=pd.DataFrame(rows)
    assert len(slots)==150
    directed_set={(r.source,r.target,int(r.lag)) for r in directed_cross.itertuples()}
    slots['discovered_directed']=[(r.source,r.target,int(r.lag)) in directed_set
                                  for r in slots.itertuples()]
    atomic_csv(slots,slots_file)
    atomic_json(slots_meta,{'run_hash':RUN_HASH,'sha256':sha_file(slots_file),
                            'rows':len(slots),'matched_rows_across_lags':True})

assert len(slots)==150 and int(slots.discovered_directed.sum())==27
assert len(set(zip(slots.source,slots.target,slots.lag)))==150
assert slots[['n_D6','n_D7']].nunique().max()==1
atomic_csv(slots,RESULTS/'06_ALL_150_CROSS_SLOTS.csv')

values=slots.weighted_delta_mse.to_numpy(np.float64)
selected=slots.discovered_directed.to_numpy(dtype=bool)
actual=float(values[selected].mean())
n=int(selected.sum())
rng=np.random.default_rng(5162026)
uniform=np.asarray([rng.choice(values,size=n,replace=False).mean()
                    for _ in range(PERMUTATIONS)])
# Control both target AND lag: sample the same count of source alternatives within each stratum.
strata=[]
for (t,lag),grp in slots.groupby(['target','lag'],sort=True):
    indices=grp.index.to_numpy(dtype=np.int64)
    count=int(selected[indices].sum())
    strata.append((indices,count))
matched=np.empty(PERMUTATIONS)
for rep in range(PERMUTATIONS):
    sample=np.concatenate([rng.choice(indices,size=count,replace=False)
                           for indices,count in strata if count])
    assert len(sample)==n
    matched[rep]=values[sample].mean()
assert np.isfinite(matched).all()
enrichment={'n_slots':int(len(slots)),'n_discovered_directed':n,
            'mean_gain_discovered':actual,
            'mean_gain_other':float(values[~selected].mean()),
            'uniform_random_set_p_descriptive':float((1+np.sum(uniform>=actual))/(PERMUTATIONS+1)),
            'target_lag_matched_set_p_descriptive':float((1+np.sum(matched>=actual))/(PERMUTATIONS+1)),
            'target_lag_matched_null_mean':float(matched.mean()),
            'target_lag_matched_null_q95':float(np.quantile(matched,.95)),
            'permutations':PERMUTATIONS,
            'caution':'These are post-discovery, D6–D7 descriptive permutation references. Slots share '
                      'autocorrelation, overlapping samples and selected conditioning. Not a causal p-value '
                      'or an independent external validation.'}
atomic_json(RESULTS/'07_SLOT_ENRICHMENT_DESCRIPTIVE.json',enrichment)
print('All-slot enrichment:',json.dumps(enrichment,indent=2))

# %%
# CELL 6 — apply historical M2 v2 tier rule; freeze sparse parent mask, no D8–D10
final=family_df.merge(support_df,on=['source','target'],how='left',validate='one_to_one')
assert len(final)==16 and final.pcmci_family_mean_support.notna().all()

def evidence_tier(r):
    robust=(r.ridge_positive_days==2 and r.ridge_weighted_delta_mse>0
            and r.hgb_weighted_delta_mse>0 and r.lag_minus_lead_gain>0
            and r.phase_bh_q<=.10)
    supported=(r.ridge_positive_days==2 and r.ridge_weighted_delta_mse>0
               and r.lag_minus_lead_gain>0
               and (r.hgb_weighted_delta_mse>0 or r.phase_bh_q<=.20))
    return ('ROBUST_PREDICTIVE_FAMILY' if robust else
            'SUPPORTED_PREDICTIVE_FAMILY' if supported else 'EXPLORATORY_FAMILY')

def orientation_flag(r):
    if r.reciprocal_directed_family:return 'CAVEATED_RECIPROCAL_FEEDBACK'
    if r.pair_has_ambiguous_endpoint:return 'CAVEATED_AMBIGUOUS_ENDPOINT'
    if r.lag_minus_lead_gain<=0:return 'CAVEATED_LEAD_PLACEBO'
    return 'CLEANER_TEMPORAL_ORIENTATION'

final['predictive_evidence_tier']=final.apply(evidence_tier,axis=1)
final['orientation_confidence']=final.apply(orientation_flag,axis=1)
primary=final[final.predictive_evidence_tier=='ROBUST_PREDICTIVE_FAMILY'].copy()
sensitivity=final[final.predictive_evidence_tier.isin([
    'ROBUST_PREDICTIVE_FAMILY','SUPPORTED_PREDICTIVE_FAMILY'])].copy()
atomic_csv(final,RESULTS/'08_FINAL_DIRECTED_FAMILY_EVIDENCE.csv')
atomic_csv(primary,RESULTS/'09_PRIMARY_ROBUST_FAMILIES.csv')
atomic_csv(sensitivity,RESULTS/'10_SENSITIVITY_FAMILIES.csv')

mask=np.zeros((6,6,6),dtype=bool)
fixed_self=[];cross=[]
for v,name in enumerate(FEATURES):
    for lag in range(1,6):
        mask[v,v,lag]=True
        fixed_self.append({'source':name,'target':name,'lag':lag,'role':'FIXED_AR5_SELF'})
for r in primary.itertuples(index=False):
    for lag in map(int,str(r.discovery_lags).split(',')):
        s=IDX[r.source];t=IDX[r.target]
        assert s!=t and str(graph[s,t,lag])=='-->'
        mask[s,t,lag]=True
        cross.append({'source':r.source,'target':r.target,'lag':lag,
                      'role':'ROBUST_DISCOVERY_CROSS',
                      'orientation_confidence':r.orientation_confidence})
assert not mask[:,:,0].any()
assert sum(mask[s,t,k] for s in range(6) for t in range(6) for k in range(1,6) if s==t)==30
assert int(mask.sum())==30+len(cross)
atomic_csv(pd.DataFrame(fixed_self),RESULTS/'11_FROZEN_SELF_AR5.csv')
atomic_csv(pd.DataFrame(cross,columns=['source','target','lag','role','orientation_confidence']),
           RESULTS/'12_FROZEN_CROSS_SLOTS.csv')
FROZEN=CKPT/'PRIMARY_FROZEN_PARENT_MASK.npz'
final_hash=sha_arrays(mask.astype(np.uint8))
if FROZEN.exists():
    with np.load(FROZEN,allow_pickle=False) as z:
        assert str(z['run_hash'].item())==RUN_HASH
        assert str(z['mask_hash'].item())==final_hash
        assert np.array_equal(z['parent_mask'],mask)
else:
    atomic_npz(FROZEN,parent_mask=mask,feature_names=np.array(FEATURES),
               run_hash=np.array(RUN_HASH),mask_hash=np.array(final_hash),
               discovery_spec_hash=np.array(meta['spec_hash']))

counts=Counter(final.predictive_evidence_tier)
summary={'stage':STAGE_VERSION,'status':'VALIDATED_AND_FROZEN_INTERNAL_ONLY',
         'run_hash':RUN_HASH,'discovery_spec_hash':meta['spec_hash'],
         'D1_D5_discovery_only':list(TRAIN_DAYS),
         'D6_D7_family_validation_only':list(VAL_DAYS),
         'D8_D10_used_for_selection':False,
         'D8_D10_historically_researcher_exposed':True,
         'directed_family_count':int(len(final)),
         'robust_families':int(counts['ROBUST_PREDICTIVE_FAMILY']),
         'supported_families':int(counts['SUPPORTED_PREDICTIVE_FAMILY']),
         'exploratory_families':int(counts['EXPLORATORY_FAMILY']),
         'self_AR5_slots':30,'cross_primary_slots':len(cross),
         'total_primary_slots':int(mask.sum()),
         'lpcmci_candidate_slots':59,'ambiguous_marks_not_oriented':5,
         'pcmci_sensitivity_only':pcmi_rows,
         'slot_enrichment_descriptive':enrichment,
         'primary_families':primary[['source','target','discovery_lags',
                                     'orientation_confidence']].to_dict(orient='records'),
         'warnings':[
             'LPCMCI edges are observational lagged candidates, NOT proven causal effects.',
             'Family tier criteria reproduce prior M2 v2 research rules; they are NOT newly preregistered.',
             'D6–D7 were inspected in prior CausalLOB work; this is reproducibility, not virgin validation.',
             'D8–D10 were also previously examined; not an external researcher-blind test.',
             'Phase-shift empirical p and slot-set permutation references are descriptive, not exact causal tests.',
             'A lead-placebo is deliberately non-deployable and cannot identify causality.',
             'No labels were accessed in structure selection; publisher labels still require separate audit.',
             'Self histories are fixed AR5, not limited to LPCMCI-discovered self slots.'
         ]}
atomic_json(RESULTS/'STEP4_FINAL_SUMMARY.json',summary)
print('\n'+'='*95)
print('STEP 4 INTERNAL VALIDATION COMPLETE — FROZEN GRAPH; NO D8–D10 SELECTION')
print('='*95)
print('Family counts:',dict(counts))
print('Primary slots: 30 self +',len(cross),'cross =',int(mask.sum()))
print('Frozen mask hash:',final_hash)
print('Primary families:\n',primary[['source','target','discovery_lags',
                                   'orientation_confidence']].to_string(index=False))
print('Output:',OUT)
print(f'Runtime: {time.time()-T0:.1f}s')
print('NEXT: independent target-label audit, then frozen parametric DBN; never retune graph on D8–D10.')

Mounted at /content/drive
STEP 4 PREFLIGHT PASS; no D8–D10 state or label used.
Step 3 discovery fingerprint: fa32b947d8ece5ff253e8784eb0ae2cfa58e068a9951841edffc0622f61e33b0
Step 4 run fingerprint: f146c37e25cbe4c084f40def8d6c8d766e3d0f1c91b9cd6c7eda218009721e33
59 candidate links / 27 directed cross slots / 16 directed families / 5 ambiguous marks.
Train D1–5: 177752 Validation D6–7: 76298
Outputs: /content/drive/MyDrive/CausalLOB/STEP4_VALIDATION_FROZEN
Finished family 01/16: imbalance_l1 -> imbalance_l2_l5; Ridge ΔMSE=+0.000237227; phase p=0.0099
Finished family 02/16: imbalance_l1 -> log_mid_return_1; Ridge ΔMSE=+0.0282089; phase p=0.0099
Finished family 03/16: imbalance_l1 -> log_relative_spread; Ridge ΔMSE=-9.15537e-05; phase p=0.8515
Finished family 04/16: imbalance_l1 -> log_total_depth_l5; Ridge ΔMSE=-1.31469e-06; phase p=0.9604
Finished family 05/16: imbalance_l1 -> quote_ofi_depth_norm; Ridge ΔMSE=+0.00469371; phase p=0.0099
Finished family 06/16: imbalance_l2_l5 -> imbalan

## Parameter learning: frozen Gaussian DBN and ordered-probit heads

In [ ]:
# %% [markdown]
# CausalLOB — Step 6: frozen parametric dynamic Bayesian network
#
# - Treat Step 4's 39-slot graph and Step 5's publisher labels as IMMUTABLE.
# - Fit transitions and five ordinal outcome heads only on D1–D7.
# - Apply horizon-specific conservative tail purges to BOTH training and evaluation.
# - Evaluate the frozen procedure on D8–D10 once, as a historically exposed retrospective assessment.
# - Report negative results and calibration; neither accuracy nor a paper trading signal establishes PnL.
# - Preserve all prior folders; this step writes only to STEP6_FROZEN_DBN.
#
# Run cells in order on Google Colab (CPU); reruns load fingerprinted checkpoints.

# %%
# CELL 1 — setup, exact source specification and durable output directory
import os, sys, json, time, hashlib, importlib.util, subprocess
from pathlib import Path

for module, package in [('scipy','scipy'),('sklearn','scikit-learn')]:
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable,'-m','pip','install','-q',package])

import numpy as np
import pandas as pd
from scipy.stats import norm
from scipy.optimize import minimize
from sklearn.linear_model import Ridge
from sklearn.metrics import balanced_accuracy_score, f1_score, log_loss

try:
    from IPython.display import display
except ImportError:
    display = print
try:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive')
    else:
        print('Google Drive already mounted.')
except ImportError:
    pass

T0=time.time()
ROOT=Path(os.environ.get('CAUSALLOB_ROOT','/content/drive/MyDrive/CausalLOB'))
S2=ROOT/'STEP2_REBUILD'/'checkpoints'
S4=ROOT/'STEP4_VALIDATION_FROZEN'
S5=ROOT/'STEP5_LABEL_AUDIT_READONLY'/'results'
OUT=ROOT/'STEP6_FROZEN_DBN'
CKPT=OUT/'checkpoints'
RESULTS=OUT/'results'
HEAD_DIR=CKPT/'outcome_heads'
for directory in (CKPT,RESULTS,HEAD_DIR):
    directory.mkdir(parents=True,exist_ok=True)

VERSION='STEP6_FROZEN_DBN_v1_2026-09-18'
FEATURES=['log_relative_spread','log_total_depth_l5','imbalance_l1',
          'imbalance_l2_l5','quote_ofi_depth_norm','log_mid_return_1']
HORIZONS=[10,20,30,50,100]
REPS=[1,2,3,5,10]
TRAIN_DAYS=[1,2,3,4,5,6,7]
EVAL_DAYS=[8,9,10]
L=5; N=6; ALPHA=1.0; ECE_BINS=15
MASK_HASH_EXPECTED='eb38b17c1431c0a98d46ed1c792953c932c58572799b64aee79165dfc58439da'

SOURCE={
 'temporal':S2/'04_temporal_primary.npz',
 'raw':S2/'01_canonical_raw.npz',
 'mask':S4/'checkpoints'/'PRIMARY_FROZEN_PARENT_MASK.npz',
 'step4_summary':S4/'results'/'STEP4_FINAL_SUMMARY.json',
 'step5_summary':S5/'05_LABEL_AUDIT_SUMMARY.json',
}
for name,path in SOURCE.items():
    if not path.is_file():
        raise FileNotFoundError(f'Missing READ-ONLY Step 6 prerequisite [{name}]: {path}')

def hash_file(path):
    digest=hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''):
            digest.update(block)
    return digest.hexdigest()

def hash_arrays(*arrays):
    digest=hashlib.sha256()
    for arr in arrays:
        x=np.ascontiguousarray(arr)
        digest.update(str(x.shape).encode())
        digest.update(str(x.dtype).encode())
        digest.update(x.view(np.uint8))
    return digest.hexdigest()

def atomic_json(path,data):
    text=json.dumps(data,sort_keys=True,indent=2,allow_nan=False)+'\n'
    path=Path(path); tmp=path.with_name(path.name+'.tmp')
    tmp.write_text(text,encoding='utf8');tmp.replace(path)

def atomic_npz(path,**arrays):
    path=Path(path);tmp=path.with_name(path.name+'.tmp.npz')
    np.savez_compressed(tmp,**arrays);tmp.replace(path)

def safe_csv(path,frame):
    path=Path(path); tmp=path.with_name(path.name+'.tmp')
    frame.to_csv(tmp,index=False,float_format='%.12g');tmp.replace(path)

print('CausalLOB Step 6 — read-only source root:',ROOT)
print('Outputs (new directory only):',OUT)

# %%
# CELL 2 — rigorous provenance, complete sequence isolation and horizon eligibility
s4=json.loads(SOURCE['step4_summary'].read_text())
s5=json.loads(SOURCE['step5_summary'].read_text())
assert s4['status']=='VALIDATED_AND_FROZEN_INTERNAL_ONLY'
assert s4['D8_D10_used_for_selection'] is False
assert s4['total_primary_slots']==39 and s4.get('cross_primary_slots',9)==9
assert s5['status']=='AUDIT_COMPLETED_WITH_LABEL_PROVENANCE_LIMITATION'
assert s5['frozen_graph_unchanged'] and s5['days_8_to_10_scored'] is False
assert s5['matched_publisher_label_array_to_temporal']
assert s5['frozen_mask_hash']==MASK_HASH_EXPECTED
assert s5['frozen_conservative_tail_purge_representation_rows_per_sequence']=={str(h):h for h in HORIZONS}
assert all(s5['old_M1_comparison'][k] for k in ['raw_exact','labels_exact','day_exact','sequence_exact'])

hashes={k:hash_file(p) for k,p in SOURCE.items()}
assert hashes['mask']==s5['input_hashes']['PRIMARY_FROZEN_PARENT_MASK.npz']
assert hashes['raw']==s5['input_hashes']['01_canonical_raw.npz']
assert hashes['temporal']==s5['input_hashes']['04_temporal_primary.npz']
assert hashes['step4_summary']==s5['input_hashes']['STEP4_FINAL_SUMMARY.json']
with np.load(SOURCE['mask'],allow_pickle=False) as z:
    mask=z['parent_mask'].copy()
    assert z['feature_names'].astype(str).tolist()==FEATURES
    assert str(z['mask_hash'].item())==MASK_HASH_EXPECTED
assert mask.shape==(6,6,6) and mask.dtype==bool and int(mask.sum())==39
assert hash_arrays(mask.astype(np.uint8))==MASK_HASH_EXPECTED
assert not mask[:,:,0].any()
assert all(mask[i,i,1:].all() for i in range(6))
assert sum(mask[s,t,l] for s in range(6) for t in range(6)
           for l in range(1,6) if s!=t)==9

with np.load(SOURCE['temporal'],allow_pickle=False) as z:
    assert str(z['checkpoint_version'].item())=='STEP2_REBUILD_v1_2026-09-17'
    assert z['feature_names'].astype(str).tolist()==FEATURES
    X=z['X_current'].astype(np.float64)
    Y=z['Y'].astype(np.int8)
    day=z['day'].astype(np.int8)
    seq=z['sequence_id'].astype(np.int16)
    lagvalid=z['full_lag_valid'].copy()
assert X.shape==(393337,6) and Y.shape==(393337,5)
assert len(day)==len(seq)==len(lagvalid)==len(X)
assert np.isfinite(X).all() and np.isin(Y,[1,2,3]).all()
assert np.array_equal(np.unique(day),np.arange(1,11))
assert len(np.unique(seq))==50 and int(lagvalid.sum())==393087
assert np.all(np.diff(day)>=0)

# Independent label array cross-check against the raw checkpoint, accounting for
# exactly 20 discarded warm-up representations per sequence.
with np.load(SOURCE['raw'],allow_pickle=False) as z:
    raw_y=z['Y'].copy()
    raw_day=z['day'].copy()
assert raw_y.shape==(394337,5) and np.array_equal(np.unique(raw_day),np.arange(1,11))
raw_seq_start=np.flatnonzero(np.r_[True,raw_day[1:]!=raw_day[:-1]])
# Raw identity is established by Step 5's exact raw/sequence audit; reconstruct
# its current-valid alignment from the original Step 2 first-20-rows rule.
with np.load(S2/'02_boundaries.npz',allow_pickle=False) as z:
    raw_seq=z['sequence_id'].copy()
raw_starts=np.flatnonzero(np.r_[True,raw_seq[1:]!=raw_seq[:-1]])
assert len(raw_starts)==50
raw_ends=np.r_[raw_starts[1:],len(raw_seq)]
raw_current=np.concatenate([np.arange(a+20,b) for a,b in zip(raw_starts,raw_ends)])
assert len(raw_current)==len(X)
assert np.array_equal(raw_y[raw_current],Y)
assert np.array_equal(raw_day[raw_current],day)
assert np.array_equal(raw_seq[raw_current],seq)
del raw_y,raw_day,raw_seq

starts=np.flatnonzero(np.r_[True,seq[1:]!=seq[:-1]])
ends=np.r_[starts[1:],len(seq)]
assert len(starts)==50
remaining=np.empty(len(seq),dtype=np.int32)
train_rows=[];eval_rows=[]
for a,b in zip(starts,ends):
    assert b-a>110 and np.all(seq[a:b]==seq[a]) and np.all(day[a:b]==day[a])
    expected_lags=np.r_[np.zeros(L,dtype=bool),np.ones(b-a-L,dtype=bool)]
    assert np.array_equal(lagvalid[a:b],expected_lags), 'Inconsistent Step2 lag validity.'
    remaining[a:b]=b-1-np.arange(a,b)
    index=np.arange(a+L,b,dtype=np.int64)
    if int(day[a]) in TRAIN_DAYS: train_rows.append(index)
    elif int(day[a]) in EVAL_DAYS: eval_rows.append(index)
train_idx=np.concatenate(train_rows);eval_idx=np.concatenate(eval_rows)
assert len(train_idx)==253875 and len(eval_idx)==139212
assert np.array_equal(np.unique(day[train_idx]),TRAIN_DAYS)
assert np.array_equal(np.unique(day[eval_idx]),EVAL_DAYS)
assert set(seq[train_idx]).isdisjoint(set(seq[eval_idx]))
assert lagvalid[train_idx].all() and lagvalid[eval_idx].all()

run_spec={'version':VERSION,'source_sha256':hashes,'frozen_mask_hash':MASK_HASH_EXPECTED,
          'train_days':TRAIN_DAYS,'assessment_days_historically_exposed':EVAL_DAYS,
          'ridge_alpha':ALPHA,'max_lag':L,'horizons':HORIZONS,
          'conservative_tail_purge_representation_rows':HORIZONS,
          'head':'six-current-state ordered probit, probit link, no intercept, analytic-gradient L-BFGS-B maxiter=160',
          'innovation_covariance':'diagonal; correlated residuals descriptive only',
          'metric_policy':'same transition rows, label tail purge per sequence',
          'selection':'none after Step4, no hyperparameter optimization on D8-D10'}
RUN_HASH=hashlib.sha256(json.dumps(run_spec,sort_keys=True).encode()).hexdigest()
manifest_path=CKPT/'IMMUTABLE_STEP6_SPEC.json'
if manifest_path.exists():
    old=json.loads(manifest_path.read_text())
    if old.get('run_hash')!=RUN_HASH:
        raise RuntimeError('Existing Step6 checkpoint has DIFFERENT inputs/specification. STOP.')
else:
    atomic_json(manifest_path,{'run_hash':RUN_HASH,'spec':run_spec})
print('PREFLIGHT PASS: Step 2/4/5 inputs exact, publisher labels unmodified, mask fixed.')
print('Train transition rows:',len(train_idx),' | retrospective assessment rows:',len(eval_idx))
print('Graph: 30 self + 9 cross =',int(mask.sum()),' | run hash:',RUN_HASH)

# %%
# CELL 3 — train-only standardization, vectorized Ridge CPDs, state-space freeze
normalization_mask=np.isin(day,TRAIN_DAYS)
mu=X[normalization_mask].mean(axis=0)
sigma=X[normalization_mask].std(axis=0,ddof=0)
assert np.isfinite(mu).all() and (sigma>1e-12).all()
XZ=(X-mu)/sigma
Ztr=np.concatenate([XZ[train_idx-lag] for lag in range(1,L+1)],axis=1)
Zev=np.concatenate([XZ[eval_idx-lag] for lag in range(1,L+1)],axis=1)
Ytr=XZ[train_idx]; Yev=XZ[eval_idx]
assert Ztr.shape==(253875,30) and Zev.shape==(139212,30)
assert np.isfinite(Ztr).all() and np.isfinite(Zev).all()

def fit_transition(parent_mask):
    B=np.zeros((6,30),dtype=float)
    intercept=np.zeros(6,dtype=float)
    variance=np.zeros(6,dtype=float)
    for target in range(6):
        parents=[(lag-1)*6+source for lag in range(1,6)
                 for source in range(6) if parent_mask[source,target,lag]]
        assert parents and len(parents)==len(set(parents))
        reg=Ridge(alpha=ALPHA,fit_intercept=True)
        reg.fit(Ztr[:,parents],Ytr[:,target])
        B[target,parents]=reg.coef_
        intercept[target]=reg.intercept_
        error=Ytr[:,target]-(Ztr[:,parents]@reg.coef_+reg.intercept_)
        variance[target]=np.mean(error*error)
    assert (variance>1e-12).all()
    assert np.all(B[~np.array([[parent_mask[j%6,t,j//6+1]
                   for j in range(30)] for t in range(6)])]==0)
    return B,intercept,variance

def state_space(B,c,v):
    F=np.zeros((30,30),dtype=float)
    F[:6,:]=B
    for lag in range(1,5):
        F[lag*6:(lag+1)*6,(lag-1)*6:lag*6]=np.eye(6)
    offset=np.zeros(30,dtype=float);offset[:6]=c
    Q=np.zeros((30,30),dtype=float);Q[:6,:6]=np.diag(v)
    return F,offset,Q

model_path=CKPT/'01_FROZEN_STATE_SPACE.npz'
ar_mask=np.zeros_like(mask)
for variable in range(6): ar_mask[variable,variable,1:]=True
if model_path.exists():
    with np.load(model_path,allow_pickle=False) as z:
        assert str(z['run_hash'].item())==RUN_HASH,'Transition checkpoint fingerprint mismatch.'
        B_ar=z['B_ar'];c_ar=z['c_ar'];v_ar=z['v_ar']
        B_sp=z['B_sparse'];c_sp=z['c_sparse'];v_sp=z['v_sparse']
        F_ar=z['F_ar'];off_ar=z['offset_ar'];Q_ar=z['Q_ar']
        F_sp=z['F_sparse'];off_sp=z['offset_sparse'];Q_sp=z['Q_sparse']
        assert np.array_equal(z['train_mean'],mu)
        assert np.array_equal(z['train_std'],sigma)
        assert np.array_equal(z['frozen_parent_mask'],mask)
    print('Reused fingerprinted Step 6 transition checkpoint.')
else:
    B_ar,c_ar,v_ar=fit_transition(ar_mask)
    B_sp,c_sp,v_sp=fit_transition(mask)
    F_ar,off_ar,Q_ar=state_space(B_ar,c_ar,v_ar)
    F_sp,off_sp,Q_sp=state_space(B_sp,c_sp,v_sp)
    atomic_npz(model_path,run_hash=np.array(RUN_HASH),
               frozen_parent_mask=mask,feature_names=np.array(FEATURES),
               train_mean=mu,train_std=sigma,
               B_ar=B_ar,c_ar=c_ar,v_ar=v_ar,
               B_sparse=B_sp,c_sparse=c_sp,v_sparse=v_sp,
               F_ar=F_ar,offset_ar=off_ar,Q_ar=Q_ar,
               F_sparse=F_sp,offset_sparse=off_sp,Q_sparse=Q_sp)
    print('Saved frozen AR(5) and sparse 39-slot Gaussian transitions.')

assert F_sp.shape==(30,30) and np.array_equal(np.diag(Q_sp)[:6],v_sp)
assert np.allclose(F_sp[6:,:-6],np.eye(24)) # exact deterministic 4-block shift
train_error=Ytr-(Ztr@B_sp.T+c_sp)
resid_cov=np.cov(train_error,rowvar=False,ddof=0)
assert np.allclose(np.diag(resid_cov),v_sp,rtol=1e-8)
parameter_rows=[]
for model_name,B in [('AR5',B_ar),('SPARSE_39',B_sp)]:
    for t in range(6):
        for lag in range(1,6):
            for s in range(6):
                if (ar_mask if model_name=='AR5' else mask)[s,t,lag]:
                    parameter_rows.append({'model':model_name,'source':FEATURES[s],
                        'target':FEATURES[t],'lag':lag,
                        'coefficient_standardized':float(B[t,(lag-1)*6+s])})
safe_csv(RESULTS/'01_FROZEN_PARAMETER_TABLE.csv',pd.DataFrame(parameter_rows))
safe_csv(RESULTS/'01_TRAIN_RESIDUAL_CORRELATION.csv',
         pd.DataFrame(np.corrcoef(train_error,rowvar=False),columns=FEATURES,index=FEATURES)
           .rename_axis('target').reset_index())
print('Transition parameters frozen; residual innovation correlations are DESCRIPTIVE, not used as Q.')

# %%
# CELL 4 — same-row, no-tuning D8–D10 transition assessment

def transition_scores(true,pred,v):
    e=true-pred
    mse=np.mean(e*e,axis=0)
    nll=0.5*(np.log(2*np.pi*v)+mse/v)
    cover=np.mean(np.abs(e)<=norm.ppf(0.95)*np.sqrt(v),axis=0)
    return mse,nll,cover

p_ar=Zev@B_ar.T+c_ar
p_sp=Zev@B_sp.T+c_sp
rows=[]
for name,pred,variance in [('AR5',p_ar,v_ar),('SPARSE_39',p_sp,v_sp)]:
    for label,where in [('POOLED',np.ones(len(eval_idx),dtype=bool))]+[
        (str(d),day[eval_idx]==d) for d in EVAL_DAYS]:
        m,nll,coverage=transition_scores(Yev[where],pred[where],variance)
        for t in range(6):
            rows.append({'model':name,'day':label,'target':FEATURES[t],
                         'n':int(where.sum()),'rmse':float(np.sqrt(m[t])),
                         'nll':float(nll[t]),'coverage90':float(coverage[t]),
                         'abs_coverage_error90':float(abs(coverage[t]-.90))})
transitions=pd.DataFrame(rows)
safe_csv(RESULTS/'02_TRANSITION_D8_D10.csv',transitions)
pooled=transitions[transitions.day=='POOLED'].pivot(index='target',columns='model',values='rmse')
pooled['rmse_gain_AR5_minus_sparse']=pooled['AR5']-pooled['SPARSE_39']
print('\nFrozen transition assessment (standardized state; positive gain means sparse improves):')
display(pooled.round(6))

# %%
# CELL 5 — fit immutable publisher-label ordered probit heads, one per horizon
# Horizon h is in original messages; 1/2/3/5/10 representation steps.
# The conservative h-REPRESENTATION-row tail purge is frozen by Step 5, not tuned.

def eligible_indices(candidate_idx,h):
    idx=candidate_idx[remaining[candidate_idx]>=h]
    assert np.isin(day[idx], TRAIN_DAYS+EVAL_DAYS).all()
    assert lagvalid[idx].all()
    return idx

def encode_publisher(y):
    assert np.isin(y,[1,2,3]).all()
    return (3-y).astype(np.int8)  # publisher 3=Down ->0; 2=Neutral ->1; 1=Up ->2

def ordered_probs(Xstate,beta,cut):
    eta=Xstate@np.asarray(beta,dtype=float)
    c1,c2=map(float,cut)
    down=norm.cdf(c1-eta)
    upto_neutral=norm.cdf(c2-eta)
    P=np.column_stack([down,upto_neutral-down,1-upto_neutral])
    assert np.isfinite(P).all() and np.min(P)>=-1e-12
    P=np.maximum(P,1e-15)
    P/=P.sum(axis=1,keepdims=True)
    return P

def ordinal_nll_gradient(theta,Z,y):
    """Exact 3-class probit log-likelihood and analytic gradient, per observation.

    Parameterization: c1 = theta[6], c2 = c1 + exp(theta[7]).
    The intercept is identified by thresholds; no separate intercept is fitted.
    """
    beta=np.asarray(theta[:6],dtype=float)
    c1=float(theta[6]);gap=float(np.exp(theta[7]));c2=c1+gap
    eta=Z@beta
    z1=c1-eta; z2=c2-eta
    cdf1=norm.cdf(z1); cdf2=norm.cdf(z2)
    sf1=norm.sf(z1); sf2=norm.sf(z2)
    p0=cdf1
    p1=np.where(z1>0,sf1-sf2,cdf2-cdf1)
    p2=sf2
    p=np.where(y==0,p0,np.where(y==1,p1,p2))
    # Clipping is solely numerical, applied equally to likelihood and gradient.
    p=np.maximum(p,1e-14)
    phi1=norm.pdf(z1);phi2=norm.pdf(z2)
    d_eta=np.where(y==0,-phi1,np.where(y==1,phi1-phi2,phi2))
    d_c1=np.where(y==0,phi1,np.where(y==1,-phi1,0.0))
    d_c2=np.where(y==2,-phi2,np.where(y==1,phi2,0.0))
    n=len(y)
    loss=float(-np.mean(np.log(p)))
    score_eta=d_eta/p
    grad_beta=-(Z.T@score_eta)/n
    grad_c1=-np.mean((d_c1+d_c2)/p)
    grad_gap=-gap*np.mean(d_c2/p)
    gradient=np.r_[grad_beta,grad_c1,grad_gap]
    if not np.isfinite(loss) or not np.isfinite(gradient).all():
        raise FloatingPointError('Nonfinite ordinal-probit loss/gradient; STOP.')
    return loss,gradient

def fit_head(h,training_indices):
    column=HORIZONS.index(h)
    y=encode_publisher(Y[training_indices,column])
    Z=XZ[training_indices]
    freq=np.bincount(y,minlength=3).astype(float)/len(y)
    assert (freq>0).all(),'Missing class in training set.'
    c1=float(norm.ppf(freq[0]))
    c2=float(norm.ppf(freq[0]+freq[1]))
    start=np.r_[np.zeros(6),c1,np.log(c2-c1)]
    fit=minimize(ordinal_nll_gradient,start,args=(Z,y),method='L-BFGS-B',
                 jac=True,options={'maxiter':160,'ftol':1e-9,'gtol':1e-6})
    if not fit.success:
        raise RuntimeError(f'Ordered probit h{h} did NOT converge: {fit.message}. '
                           'STOP; never tune using D8–D10.')
    beta=fit.x[:6]; thresholds=[float(fit.x[6]),float(fit.x[6]+np.exp(fit.x[7]))]
    if not thresholds[0]<thresholds[1]:
        raise RuntimeError(f'Invalid ordered thresholds at h{h}.')
    return {'run_hash':RUN_HASH,'horizon_raw_messages':h,
            'horizon_representation_steps':REPS[column],
            'n_train':int(len(training_indices)),
            'train_index_sha256':hash_arrays(training_indices),
            'feature_names':FEATURES,'beta':beta.tolist(),
            'thresholds':thresholds,'converged':bool(fit.success),
            'llf':float(-len(y)*fit.fun),
            'optimizer':'analytic-gradient L-BFGS-B','iterations':int(fit.nit)}

heads={}
for h in HORIZONS:
    tr=eligible_indices(train_idx,h)
    hfile=HEAD_DIR/f'ordered_probit_h{h}.json'
    if hfile.exists():
        head=json.loads(hfile.read_text())
        assert head['run_hash']==RUN_HASH
        assert head['train_index_sha256']==hash_arrays(tr)
        assert head['n_train']==len(tr) and head['feature_names']==FEATURES
        assert head['converged'] and len(head['beta'])==6
        print(f'Reused immutable ordered-probit h{h} checkpoint.')
    else:
        print(f'Fitting frozen ordered-probit h{h}; eligible train rows={len(tr):,} ...')
        hstart=time.time()
        head=fit_head(h,tr)
        atomic_json(hfile,head)
        print(f'Saved h{h} in {time.time()-hstart:.1f}s.')
    heads[h]=head

# %%
# CELL 6 — frozen retrospective price-state assessment and inspectable probabilities

def ece(y,P,bins=ECE_BINS):
    conf=P.max(axis=1); pred=P.argmax(axis=1); hit=(pred==y)
    limits=np.linspace(0,1,bins+1)
    acc=0.0
    for i in range(bins):
        m=(conf>=limits[i]) & (conf < limits[i+1] if i<bins-1 else conf<=limits[i+1])
        if m.any():acc+=m.mean()*abs(hit[m].mean()-conf[m].mean())
    return float(acc)

def classification_metrics(y,P):
    pred=np.argmax(P,axis=1)
    brier=np.mean(np.sum((P-np.eye(3)[y])**2,axis=1))
    return {'n':int(len(y)),'accuracy':float(np.mean(pred==y)),
            'balanced_accuracy':float(balanced_accuracy_score(y,pred)),
            'macro_f1':float(f1_score(y,pred,labels=[0,1,2],average='macro',zero_division=0)),
            'log_loss':float(log_loss(y,P,labels=[0,1,2])),
            'brier_multiclass':float(brier),'ece_confidence':ece(y,P)}

classification=[]
for h in HORIZONS:
    tr=eligible_indices(train_idx,h)
    te=eligible_indices(eval_idx,h)
    column=HORIZONS.index(h)
    assert set(seq[tr]).isdisjoint(set(seq[te]))
    yt=encode_publisher(Y[tr,column]);ye=encode_publisher(Y[te,column])
    prior=np.bincount(yt,minlength=3).astype(float)
    prior/=prior.sum()
    P=ordered_probs(XZ[te],heads[h]['beta'],heads[h]['thresholds'])
    Pprior=np.broadcast_to(prior,P.shape)
    outfile=RESULTS/f'03_PREDICTED_CLASS_PROBABILITIES_h{h}.npz'
    # Full array stores labels and row identity. It is assessment-only; never use for tuning.
    if outfile.exists():
        with np.load(outfile,allow_pickle=False) as z:
            assert str(z['run_hash'].item())==RUN_HASH
            assert np.array_equal(z['row_index'],te)
            assert np.array_equal(z['p_down_neutral_up'],P), 'Prediction drift on rerun.'
    else:
        atomic_npz(outfile,run_hash=np.array(RUN_HASH),
                   row_index=te,day=day[te],sequence_id=seq[te],
                   original_label=Y[te,column],label_ordered=ye,
                   p_down_neutral_up=P,prior_down_neutral_up=prior,
                   class_names=np.array(['Down','Neutral','Up']),
                   horizon_raw_messages=np.array(h),
                   horizon_representation_steps=np.array(REPS[column]))
    for label,m in [('POOLED',np.ones(len(te),dtype=bool))]+[(str(d),day[te]==d) for d in EVAL_DAYS]:
        for name,pred in [('PRIOR_ONLY',Pprior),('ORDERED_PROBIT_SPARSE_DBN_HEAD',P)]:
            classification.append({'horizon_raw_messages':h,
                'horizon_representation_steps':REPS[column],
                'day':label,'model':name,**classification_metrics(ye[m],pred[m])})
classification=pd.DataFrame(classification)
safe_csv(RESULTS/'04_OUTCOME_METRICS_D8_D10.csv',classification)
print('\nD8–D10 retrospective assessment; Day 8–10 historically inspected in older research:')
display(classification[classification.day=='POOLED'][[
    'horizon_raw_messages','model','n','accuracy','balanced_accuracy',
    'macro_f1','log_loss','brier_multiclass','ece_confidence']].round(6))
print('\nExample h10 output:')
with np.load(RESULTS/'03_PREDICTED_CLASS_PROBABILITIES_h10.npz',allow_pickle=False) as z:
    preview=pd.DataFrame(z['p_down_neutral_up'][:5],columns=['P_Down','P_Neutral','P_Up'])
    preview.insert(0,'day',z['day'][:5])
    preview.insert(1,'publisher_label',z['original_label'][:5])
display(preview.round(6))

# %%
# CELL 7 — frozen final manifest; DO NOT claim trading profitability
summary={'version':VERSION,'status':'PARAMETRIC_DBN_FITTED_AND_ASSESSED_RETROSPECTIVELY',
         'run_hash':RUN_HASH,'frozen_mask_hash':MASK_HASH_EXPECTED,
         'structural_slots':39,'self_slots':30,'cross_slots':9,
         'train_days':TRAIN_DAYS,'retrospective_assessment_days':EVAL_DAYS,
         'train_transition_rows':int(len(train_idx)),
         'assessment_transition_rows':int(len(eval_idx)),
         'transition_models':['AR5','SPARSE_39'],
         'innovation_Q':'diagonal Gaussian primary; no contemporaneous DAG asserted',
         'outcome_heads':HORIZONS,
         'label_alignment':'Publisher original labels; 20 warmups and h representation-row tail purges',
         'model_selection_after_step4':False,
         'historical_researcher_blind_assessment':False,
         'label_generation_exactly_reconstructed':False,
         'execution_costs_measured':False,'realized_trading_profit_estimated':False,
         'limitations':[
            'D6–D7 graph selection and D8–D10 earlier researcher exposure limit inferential independence.',
            'FI-2010 supplied classification labels do not encode executable future profits.',
            'No fees, spreads on completed round trips, fill/queue position, slippage or latency simulated.',
            'A fixed directed cross-lag graph is not identified intervention-level causality.',
            'Ordered probit predicts conditional label probabilities, not conditional return magnitudes.',
            'Correlated contemporaneous residuals invalidate a strict independent-CPD interpretation if full Q is used.'
         ],'next_step':'Exact missing-state Gaussian inference with the frozen sparse DBN; covariance sensitivity separate.'}
atomic_json(RESULTS/'STEP6_FINAL_SUMMARY.json',summary)
print('\n'+'='*95)
print('STEP 6 COMPLETE — FROZEN GAUSSIAN DBN + FIVE ORDERED-PROBIT HEADS')
print('='*95)
print('All output artifacts:',RESULTS)
print('No PnL claim: accuracy is not net returns.')
print('Runtime:',round(time.time()-T0,1),'seconds')

Google Drive already mounted.
CausalLOB Step 6 — read-only source root: /content/drive/MyDrive/CausalLOB
Outputs (new directory only): /content/drive/MyDrive/CausalLOB/STEP6_FROZEN_DBN
PREFLIGHT PASS: Step 2/4/5 inputs exact, publisher labels unmodified, mask fixed.
Train transition rows: 253875  | retrospective assessment rows: 139212
Graph: 30 self + 9 cross = 39  | run hash: 9831f8e1db702848b8588a535088f9392c32e9a09b66eb5cafe40fc6d4bd1d8c
Saved frozen AR(5) and sparse 39-slot Gaussian transitions.
Transition parameters frozen; residual innovation correlations are DESCRIPTIVE, not used as Q.

Frozen transition assessment (standardized state; positive gain means sparse improves):


model,AR5,SPARSE_39,rmse_gain_AR5_minus_sparse
target,,,
imbalance_l1,0.561755,0.561755,0.000000
imbalance_l2_l5,0.406754,0.406754,0.000000
log_mid_return_1,0.833930,0.824720,0.009211
log_relative_spread,0.434254,0.434556,-0.000302
log_total_depth_l5,0.109082,0.109082,0.000000
quote_ofi_depth_norm,0.836648,0.836316,0.000332


Fitting frozen ordered-probit h10; eligible train rows=253,525 ...
Saved h10 in 6.2s.
Fitting frozen ordered-probit h20; eligible train rows=253,175 ...
Saved h20 in 2.6s.
Fitting frozen ordered-probit h30; eligible train rows=252,825 ...
Saved h30 in 2.4s.
Fitting frozen ordered-probit h50; eligible train rows=252,125 ...
Saved h50 in 1.9s.
Fitting frozen ordered-probit h100; eligible train rows=250,375 ...
Saved h100 in 2.1s.

D8–D10 retrospective assessment; Day 8–10 historically inspected in older research:


,horizon_raw_messages,model,n,accuracy,balanced_accuracy,macro_f1,log_loss,brier_multiclass,ece_confidence
0,10,PRIOR_ONLY,139062,0.707008,0.333333,0.276120,0.830344,0.472694,0.101577
1,10,ORDERED_PROBIT_SPARSE_DBN_HEAD,139062,0.877493,0.727693,0.797387,0.499358,0.237289,0.088780
8,20,PRIOR_ONLY,138912,0.620976,0.333333,0.255392,0.958605,0.566575,0.126579
9,20,ORDERED_PROBIT_SPARSE_DBN_HEAD,138912,0.781228,0.623119,0.682000,0.715190,0.386840,0.135478
16,30,PRIOR_ONLY,138762,0.561595,0.333333,0.239753,1.030661,0.619175,0.143100
17,30,ORDERED_PROBIT_SPARSE_DBN_HEAD,138762,0.685137,0.538449,0.565151,0.937313,0.540732,0.195836
24,50,PRIOR_ONLY,138462,0.274624,0.333333,0.143637,1.109434,0.673787,0.068967
25,50,ORDERED_PROBIT_SPARSE_DBN_HEAD,138462,0.322327,0.407045,0.281089,1.070878,0.643074,0.103171
32,100,PRIOR_ONLY,137712,0.342664,0.333333,0.170141,1.160453,0.700311,0.063303
33,100,ORDERED_PROBIT_SPARSE_DBN_HEAD,137712,0.369982,0.374541,0.296589,1.146834,0.690095,0.086855



Example h10 output:


,day,publisher_label,P_Down,P_Neutral,P_Up
0,8,3,0.791804,0.207566,0.000631
1,8,3,0.106839,0.771845,0.121316
2,8,1,0.000661,0.211368,0.787972
3,8,1,0.000000,0.000000,1.000000
4,8,2,0.222088,0.728111,0.049801



STEP 6 COMPLETE — FROZEN GAUSSIAN DBN + FIVE ORDERED-PROBIT HEADS
All output artifacts: /content/drive/MyDrive/CausalLOB/STEP6_FROZEN_DBN/results
No PnL claim: accuracy is not net returns.
Runtime: 28.6 seconds


## Inference: exact Gaussian conditioning under partial observation

In [ ]:
# CausalLOB Step 7 — exact partial-evidence inference; one-cell Colab-compatible script.
# frozen graph/head parameters and train-only covariance sensitivity. Report failures;
# do not tune on D8–D10, infer causality, or claim executable profits.

import os, json, time, hashlib, importlib.util, subprocess, sys
from pathlib import Path

for module, package in [('scipy','scipy'),('sklearn','scikit-learn')]:
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable,'-m','pip','install','-q',package])

import numpy as np
import pandas as pd
from scipy.stats import norm
from sklearn.covariance import LedoitWolf
from sklearn.metrics import balanced_accuracy_score, f1_score

try:
    from IPython.display import display
except ImportError:
    display = print
try:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive')
    else:
        print('Google Drive already mounted.')
except ImportError:
    pass

T0=time.time()
ROOT=Path(os.environ.get('CAUSALLOB_ROOT','/content/drive/MyDrive/CausalLOB'))
S2=ROOT/'STEP2_REBUILD'/'checkpoints'
S4=ROOT/'STEP4_VALIDATION_FROZEN'/'checkpoints'
S5=ROOT/'STEP5_LABEL_AUDIT_READONLY'/'results'
S6=ROOT/'STEP6_FROZEN_DBN'
OUT=ROOT/'STEP7_MISSING_STATE_READONLY'
CKPT=OUT/'checkpoints'; RESULTS=OUT/'results'; PROBS=RESULTS/'focal_probabilities'
VERSION='STEP7_EXACT_MISSING_STATE_v1_2026-09-18'
EXPECTED_MASK_HASH='eb38b17c1431c0a98d46ed1c792953c932c58572799b64aee79165dfc58439da'
FEATURES=['log_relative_spread','log_total_depth_l5','imbalance_l1',
          'imbalance_l2_l5','quote_ofi_depth_norm','log_mid_return_1']
HORIZONS=[10,20,30,50,100]
TRAIN_DAYS=[1,2,3,4,5,6,7]
EVAL_DAYS=[8,9,10]
MASK_SCENARIOS={
    'FULL_OBSERVED':[],
    'MISS_L1_IMBALANCE':['imbalance_l1'],
    'MISS_OFI_PROXY':['quote_ofi_depth_norm'],
    'MISS_BOTH_IMBALANCES':['imbalance_l1','imbalance_l2_l5'],
    'MISS_LIQUIDITY':['log_relative_spread','log_total_depth_l5'],
    'MISS_RETURN_AND_OFI':['log_mid_return_1','quote_ofi_depth_norm'],
    'HISTORY_ONLY':FEATURES.copy(),
}
FOCAL_SCENARIOS=['MISS_RETURN_AND_OFI','HISTORY_ONLY']
FOCAL_HORIZONS=[10,20]
MASK_IDX={name:np.array(sorted(FEATURES.index(x) for x in vars_),dtype=np.int64)
          for name,vars_ in MASK_SCENARIOS.items()}

SOURCE={
 'temporal':S2/'04_temporal_primary.npz',
 'step4_mask':S4/'PRIMARY_FROZEN_PARENT_MASK.npz',
 'step5_summary':S5/'05_LABEL_AUDIT_SUMMARY.json',
 'step6_state':S6/'checkpoints'/'01_FROZEN_STATE_SPACE.npz',
 'step6_summary':S6/'results'/'STEP6_FINAL_SUMMARY.json',
 'step6_outcome_metrics':S6/'results'/'04_OUTCOME_METRICS_D8_D10.csv',
}
for h in HORIZONS:
    SOURCE[f'head_h{h}']=S6/'checkpoints'/'outcome_heads'/f'ordered_probit_h{h}.json'
    SOURCE[f'pred_h{h}']=S6/'results'/f'03_PREDICTED_CLASS_PROBABILITIES_h{h}.npz'


def sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''):
            h.update(block)
    return h.hexdigest()


def atomic_json(path,obj):
    path=Path(path); tmp=path.with_name(path.name+'.tmp')
    tmp.write_text(json.dumps(obj,sort_keys=True,indent=2,allow_nan=False)+'\n',encoding='utf8')
    tmp.replace(path)


def atomic_npz(path,**kwargs):
    path=Path(path);tmp=path.with_name(path.name+'.tmp.npz')
    np.savez_compressed(tmp,**kwargs);tmp.replace(path)


def atomic_csv(path,df):
    path=Path(path);tmp=path.with_name(path.name+'.tmp')
    df.to_csv(tmp,index=False,float_format='%.12g');tmp.replace(path)


def assert_spd(Q,name):
    assert np.allclose(Q,Q.T,rtol=1e-12,atol=1e-12),f'{name} not symmetric'
    ev=np.linalg.eigvalsh(Q)
    assert np.isfinite(ev).all() and ev.min()>1e-12,f'{name} covariance not SPD'
    return ev


def reference_ordered_probs(Z,beta,cut):
    """Identical to frozen Step 6 head evaluation, for strict parity."""
    eta=Z@np.asarray(beta,dtype=float)
    c1,c2=map(float,cut)
    down=norm.cdf(c1-eta)
    upto_neutral=norm.cdf(c2-eta)
    P=np.column_stack([down,upto_neutral-down,1-upto_neutral])
    assert np.isfinite(P).all() and np.min(P)>=-1e-12
    P=np.maximum(P,1e-15)
    P/=P.sum(axis=1,keepdims=True)
    return P


def condition_exact(prior,actual,Q,missing):
    """One-step p(X_t,miss | exact X_t,obs, exact X_{t-1:t-5}).

    Observed components are *copied* from observed input. Their posterior variance is 0.
    No evaluation-period parameter fitting or future-state smoothing is involved.
    """
    missing=np.asarray(missing,dtype=int)
    observed=np.array([j for j in range(6) if j not in missing],dtype=int)
    n=len(prior)
    assert prior.shape==actual.shape==(n,6)
    post=actual.copy()
    Sfull=np.zeros((6,6),dtype=np.float64)
    if not len(missing):
        return post,Sfull,np.zeros((0,0))
    if not len(observed):
        Sfull=Q.copy()
        return prior.copy(),Sfull,Q.copy()
    Soo=Q[np.ix_(observed,observed)]
    Smo=Q[np.ix_(missing,observed)]
    Smm=Q[np.ix_(missing,missing)]
    # Positive definiteness asserted; solve, not pseudoinverse, avoids hiding defects.
    K=np.linalg.solve(Soo,Smo.T).T
    post[:,missing]=prior[:,missing]+(actual[:,observed]-prior[:,observed])@K.T
    Scond=Smm-K@Smo.T
    Scond=(Scond+Scond.T)*0.5
    assert_spd(Scond,'posterior missing state')
    Sfull[np.ix_(missing,missing)]=Scond
    assert np.array_equal(post[:,observed],actual[:,observed])
    return post,Sfull,Scond


def state_metrics(actual,post,Scond,missing):
    missing=np.asarray(missing,dtype=int)
    if not len(missing):
        return {'state_rmse':0.0,'state_mae':0.0,'state_joint_nll':np.nan,
                'coverage90':np.nan,'abs_coverage_error90':np.nan,
                'mean_interval_width90':np.nan}
    err=actual[:,missing]-post[:,missing]
    rmse=float(np.sqrt(np.mean(err**2)))
    mae=float(np.mean(np.abs(err)))
    eig=assert_spd(Scond,'state score covariance')
    sign,logdet=np.linalg.slogdet(Scond)
    assert sign>0
    whitened=np.linalg.solve(Scond,err.T).T
    quad=np.einsum('ij,ij->i',err,whitened)
    assert np.isfinite(quad).all() and quad.min()>-1e-8
    d=len(missing)
    nll=float(np.mean(0.5*(d*np.log(2*np.pi)+logdet+quad)))
    halfwidth=norm.ppf(.95)*np.sqrt(np.diag(Scond))
    cov=float(np.mean(np.abs(err)<=halfwidth))
    return {'state_rmse':rmse,'state_mae':mae,'state_joint_nll':nll,
            'coverage90':cov,'abs_coverage_error90':abs(cov-.90),
            'mean_interval_width90':float(np.mean(2*halfwidth))}


def probs_integrated(post,Sfull,head):
    beta=np.asarray(head['beta'],dtype=np.float64)
    c1,c2=map(float,head['thresholds'])
    assert beta.shape==(6,) and c1<c2
    extra=float(beta@Sfull@beta)
    assert extra>=-1e-12
    sd=np.sqrt(1.0+max(0.,extra))
    eta=post@beta
    below=norm.cdf((c1-eta)/sd)
    upto=norm.cdf((c2-eta)/sd)
    P=np.column_stack([below,upto-below,1.0-upto])
    assert np.isfinite(P).all() and P.min()>=-1e-12
    P=np.maximum(P,1e-15)
    P/=P.sum(axis=1,keepdims=True)
    return P,extra


def metrics(y,P):
    y=np.asarray(y,dtype=int)
    assert np.isin(y,[0,1,2]).all() and P.shape==(len(y),3)
    assert np.allclose(P.sum(axis=1),1.,atol=1e-12)
    pred=P.argmax(axis=1)
    n=len(y)
    chosen=P[np.arange(n),y]
    ll=float(-np.mean(np.log(np.maximum(chosen,1e-15))))
    brier=float(np.mean(np.sum((P-np.eye(3)[y])**2,axis=1)))
    conf=P.max(axis=1); correct=pred==y; ece=0.
    edges=np.linspace(0,1,16)
    for j in range(15):
        inside=(conf>=edges[j]) & ((conf<=edges[j+1]) if j==14 else (conf<edges[j+1]))
        if inside.any():
            ece+=inside.mean()*abs(correct[inside].mean()-conf[inside].mean())
    return {'n':n,'accuracy':float(correct.mean()),
            'balanced_accuracy':float(balanced_accuracy_score(y,pred)),
            'macro_f1':float(f1_score(y,pred,labels=[0,1,2],average='macro',zero_division=0)),
            'log_loss':ll,'brier_multiclass':brier,'ece_confidence':float(ece)}


def eligible(indices,remaining,h):
    return indices[remaining[indices]>=h]


# Local function-level analytic checks; these run before using real checkpoints.
def self_test():
    rng=np.random.default_rng(5162026)
    prior=rng.normal(size=(17,6));actual=rng.normal(size=(17,6))
    A=rng.normal(size=(6,6));Q=A@A.T+np.eye(6)*.2
    for missing in [[],[2],[4,5],[0,1,2,3,4,5]]:
        post,Sfull,S=condition_exact(prior,actual,Q,missing)
        observed=[j for j in range(6) if j not in missing]
        if observed:
            assert np.array_equal(post[:,observed],actual[:,observed])
            assert np.all(Sfull[np.ix_(observed,observed)]==0)
        if len(missing)==6:
            assert np.array_equal(post,prior) and np.array_equal(S,Q)
        if not missing:
            assert np.array_equal(post,actual) and np.count_nonzero(Sfull)==0
        else:
            z=state_metrics(actual,post,S,missing)
            assert np.isfinite(z['state_joint_nll'])
    diag=np.diag(np.arange(1.,7.))
    p,Sfull,S=condition_exact(prior,actual,diag,[4,5])
    assert np.array_equal(p[:,[4,5]],prior[:,[4,5]])
    dummy={'beta':np.array([.2,-.1,.4,0,.3,.5]),'thresholds':[-.3,.7]}
    P,e=probs_integrated(actual,np.zeros((6,6)),dummy)
    assert np.array_equal(P,reference_ordered_probs(actual,dummy['beta'],dummy['thresholds']))
    assert np.array_equal(P.sum(axis=1).round(12),np.ones(17))
    assert e==0.
    P2,e2=probs_integrated(p,Sfull,dummy)
    assert e2>0 and np.isfinite(P2).all()
    print('SELF-TEST PASS: zero/full/partial masks, SPD, diagonal no cross-conditioning, ordinal integration')

self_test()

for name,path in SOURCE.items():
    if not path.is_file():
        raise FileNotFoundError(f'Missing READ-ONLY prerequisite [{name}]: {path}')

source_hashes={name:sha256(path) for name,path in SOURCE.items()}
s5=json.loads(SOURCE['step5_summary'].read_text())
s6=json.loads(SOURCE['step6_summary'].read_text())
assert s5['frozen_mask_hash']==EXPECTED_MASK_HASH
assert s6['status']=='PARAMETRIC_DBN_FITTED_AND_ASSESSED_RETROSPECTIVELY'
assert s6['frozen_mask_hash']==EXPECTED_MASK_HASH
assert s6['structural_slots']==39 and s6['model_selection_after_step4'] is False
assert s6['historical_researcher_blind_assessment'] is False
with np.load(SOURCE['step4_mask'],allow_pickle=False) as z:
    frozen_mask=z['parent_mask'].copy()
    assert str(z['mask_hash'].item())==EXPECTED_MASK_HASH
    assert z['feature_names'].astype(str).tolist()==FEATURES
assert frozen_mask.shape==(6,6,6) and frozen_mask.sum()==39
with np.load(SOURCE['temporal'],allow_pickle=False) as z:
    assert str(z['checkpoint_version'].item())=='STEP2_REBUILD_v1_2026-09-17'
    assert z['feature_names'].astype(str).tolist()==FEATURES
    X=z['X_current'].astype(np.float64)
    Y=z['Y'].astype(np.int8)
    day=z['day'].astype(np.int8)
    seq=z['sequence_id'].astype(np.int16)
    lagvalid=z['full_lag_valid'].astype(bool)
assert X.shape==(393337,6) and Y.shape==(393337,5)
assert len(np.unique(seq))==50 and np.isfinite(X).all()
assert lagvalid.sum()==393087 and np.isin(Y,[1,2,3]).all()
assert np.all(np.diff(day)>=0)
with np.load(SOURCE['step6_state'],allow_pickle=False) as z:
    assert str(z['run_hash'].item())==s6['run_hash']
    assert z['feature_names'].astype(str).tolist()==FEATURES
    assert np.array_equal(z['frozen_parent_mask'],frozen_mask)
    mu=z['train_mean'].copy();sigma=z['train_std'].copy()
    B_ar=z['B_ar'].copy();c_ar=z['c_ar'].copy();v_ar=z['v_ar'].copy()
    B_sp=z['B_sparse'].copy();c_sp=z['c_sparse'].copy();v_sp=z['v_sparse'].copy()
    F_sp=z['F_sparse'].copy();Q_sp=z['Q_sparse'].copy()
assert B_ar.shape==B_sp.shape==(6,30) and c_ar.shape==c_sp.shape==(6,)
assert np.allclose(F_sp[:6],B_sp) and np.allclose(Q_sp[:6,:6],np.diag(v_sp))
assert np.array_equal(mu,X[np.isin(day,TRAIN_DAYS)].mean(axis=0))
assert np.array_equal(sigma,X[np.isin(day,TRAIN_DAYS)].std(axis=0,ddof=0))
assert (sigma>0).all()
heads={}
for h in HORIZONS:
    head=json.loads(SOURCE[f'head_h{h}'].read_text())
    assert head['run_hash']==s6['run_hash'] and head['converged']
    assert head['feature_names']==FEATURES and len(head['beta'])==6
    heads[h]=head

starts=np.flatnonzero(np.r_[True,seq[1:]!=seq[:-1]])
ends=np.r_[starts[1:],len(seq)]
remaining=np.empty(len(seq),dtype=np.int32)
train_rows=[];eval_rows=[]
for a,b in zip(starts,ends):
    assert b-a>110 and np.all(seq[a:b]==seq[a]) and np.all(day[a:b]==day[a])
    expected_lags=np.r_[np.zeros(5,dtype=bool),np.ones(b-a-5,dtype=bool)]
    assert np.array_equal(lagvalid[a:b],expected_lags)
    remaining[a:b]=b-1-np.arange(a,b)
    indices=np.arange(a+5,b,dtype=np.int64)
    if int(day[a]) in TRAIN_DAYS:train_rows.append(indices)
    elif int(day[a]) in EVAL_DAYS:eval_rows.append(indices)
train_idx=np.concatenate(train_rows);eval_idx=np.concatenate(eval_rows)
assert len(train_idx)==253875 and len(eval_idx)==139212
assert set(seq[train_idx]).isdisjoint(set(seq[eval_idx]))
assert np.isin(day[train_idx],TRAIN_DAYS).all()
assert np.isin(day[eval_idx],EVAL_DAYS).all()

# Verify all original Step 6 probability arrays and exactly matched row identities.
XZ=(X-mu)/sigma
reference={}
for h in HORIZONS:
    idx=eligible(eval_idx,remaining,h)
    with np.load(SOURCE[f'pred_h{h}'],allow_pickle=False) as z:
        assert str(z['run_hash'].item())==s6['run_hash']
        assert np.array_equal(z['row_index'],idx)
        assert np.array_equal(z['sequence_id'],seq[idx])
        assert np.array_equal(z['day'],day[idx])
        assert np.array_equal(z['original_label'],Y[idx,HORIZONS.index(h)])
        pref=z['p_down_neutral_up'].copy()
    pcheck=reference_ordered_probs(XZ[idx],heads[h]['beta'],heads[h]['thresholds'])
    assert np.array_equal(pref,pcheck),f'Head h{h}: not exact Step 6 parity!'
    reference[h]=pref

spec={
 'version':VERSION,'source_sha256':source_hashes,
 'step6_run_hash':s6['run_hash'],'frozen_mask_hash':EXPECTED_MASK_HASH,
 'train_days':TRAIN_DAYS,'retrospective_days_historically_exposed':EVAL_DAYS,
 'missing_evidence_masks':MASK_SCENARIOS,'horizons':HORIZONS,
 'transition':'read frozen AR5 and sparse39, no new transition fitting',
 'covariances':'Step6 frozen diagonal primary; LedoitWolf training-residual shrinkage sensitivity',
 'inference':'exact one-step Gaussian conditioning, full exact history and noiseless observed state',
 'outcome':'analytically integrated immutable Step6 ordered probit heads',
 'tail_purge':'h representation rows per sequence, inherited Step5/6 conservative policy',
 'model_selection':'none; no D8–D10 parameter fitting or hyperparameter/graph/head selection',
}
run_hash=hashlib.sha256(json.dumps(spec,sort_keys=True).encode()).hexdigest()
for p in [CKPT,RESULTS,PROBS]:p.mkdir(parents=True,exist_ok=True)
immutable=CKPT/'IMMUTABLE_STEP7_SPEC.json'
if immutable.exists():
    prior_spec=json.loads(immutable.read_text())
    assert prior_spec['run_hash']==run_hash,'Existing Step7 outputs have different inputs/spec! STOP.'
else:
    atomic_json(immutable,{'run_hash':run_hash,'spec':spec})
print('PREFLIGHT PASS: frozen 39-slot graph and Step6 h10...h100 outputs reproduced bitwise.')
print('Train-only residual rows:',len(train_idx),' | D8–D10 retrospective rows:',len(eval_idx))
print('Run hash:',run_hash)

# Training-only covariance estimation, sensitive to contemporaneous residual correlations.
Ztr=np.concatenate([XZ[train_idx-lag] for lag in range(1,6)],axis=1)
Zev=np.concatenate([XZ[eval_idx-lag] for lag in range(1,6)],axis=1)
Ytr=XZ[train_idx];Yev=XZ[eval_idx]
assert Ztr.shape==(253875,30) and Zev.shape==(139212,30)
variants={}
for name,B,c,v in [('AR5',B_ar,c_ar,v_ar),('SPARSE_39',B_sp,c_sp,v_sp)]:
    residual=Ytr-(Ztr@B.T+c)
    assert np.allclose(np.mean(residual*residual,axis=0),v,rtol=5e-10,atol=5e-10)
    lw=LedoitWolf().fit(residual)
    for covname,Q,shrinkage in [
        ('DIAG_FROZEN',np.diag(v),None),
        ('SHRINK_TRAIN_SENSITIVITY',lw.covariance_,float(lw.shrinkage_)),
    ]:
        tag=name+'__'+covname
        ev=assert_spd(Q,tag)
        variants[tag]={'B':B,'c':c,'Q':Q,'structure':name,'covariance':covname,
                       'shrinkage':shrinkage,'eigenvalues':ev}

cov_rows=[]
for name,d in variants.items():
    Q=d['Q'];sd=np.sqrt(np.diag(Q));corr=Q/np.outer(sd,sd)
    cov_rows.append({'model':name,'structure':d['structure'],'covariance':d['covariance'],
                     'shrinkage_train_only':d['shrinkage'],
                     'min_eigenvalue':float(d['eigenvalues'].min()),
                     'condition_number':float(d['eigenvalues'].max()/d['eigenvalues'].min()),
                     'max_abs_offdiag_correlation':float(np.max(np.abs(corr-np.eye(6))))})
atomic_csv(RESULTS/'01_TRAIN_ONLY_COVARIANCE_DIAGNOSTICS.csv',pd.DataFrame(cov_rows))
if not (CKPT/'01_TRAIN_ONLY_COVARIANCES.npz').exists():
    atomic_npz(CKPT/'01_TRAIN_ONLY_COVARIANCES.npz',run_hash=np.array(run_hash),
        Q_ar_diag=variants['AR5__DIAG_FROZEN']['Q'],
        Q_sparse_diag=variants['SPARSE_39__DIAG_FROZEN']['Q'],
        Q_ar_shrink=variants['AR5__SHRINK_TRAIN_SENSITIVITY']['Q'],
        Q_sparse_shrink=variants['SPARSE_39__SHRINK_TRAIN_SENSITIVITY']['Q'])
else:
    with np.load(CKPT/'01_TRAIN_ONLY_COVARIANCES.npz',allow_pickle=False) as z:
        assert str(z['run_hash'].item())==run_hash
        assert np.array_equal(z['Q_sparse_diag'],variants['SPARSE_39__DIAG_FROZEN']['Q'])
        assert np.array_equal(z['Q_sparse_shrink'],variants['SPARSE_39__SHRINK_TRAIN_SENSITIVITY']['Q'])
        assert np.array_equal(z['Q_ar_diag'],variants['AR5__DIAG_FROZEN']['Q'])
        assert np.array_equal(z['Q_ar_shrink'],variants['AR5__SHRINK_TRAIN_SENSITIVITY']['Q'])
print('Train-only innovation covariances ready. Shrinkage is sensitivity, NOT a replacement for frozen diagonal DBN.')

# Exact inference and identical D8–D10 subsets for each variant and missingness scenario.
state_rows=[];outcome_rows=[]
for name,d in variants.items():
    prior=Zev@d['B'].T+d['c']
    for scenario,missing in MASK_IDX.items():
        posterior,Sfull,Scond=condition_exact(prior,Yev,d['Q'],missing)
        for group,m in [('POOLED',np.ones(len(eval_idx),dtype=bool))]+[
            (str(daynum),day[eval_idx]==daynum) for daynum in EVAL_DAYS]:
            state_rows.append({'model':name,'structure':d['structure'],
                'covariance':d['covariance'],'scenario':scenario,'day':group,
                'n':int(m.sum()),'n_missing':len(missing),
                **state_metrics(Yev[m],posterior[m],Scond,missing)})
        for h in HORIZONS:
            idx=eligible(eval_idx,remaining,h)
            loc=np.searchsorted(eval_idx,idx)
            assert np.array_equal(eval_idx[loc],idx)
            y=(3-Y[idx,HORIZONS.index(h)]).astype(int)
            if not len(missing):
                P=reference_ordered_probs(posterior[loc],heads[h]['beta'],heads[h]['thresholds'])
                assert np.array_equal(P,reference[h]),f'Full-observed Step6 drift h{h}'
                latent_extra=0.0
            else:
                P,latent_extra=probs_integrated(posterior[loc],Sfull,heads[h])
            for group,m in [('POOLED',np.ones(len(idx),dtype=bool))]+[
                (str(daynum),day[idx]==daynum) for daynum in EVAL_DAYS]:
                outcome_rows.append({'model':name,'structure':d['structure'],
                    'covariance':d['covariance'],'scenario':scenario,'horizon':h,
                    'day':group,'latent_extra_variance':latent_extra,**metrics(y[m],P[m])})
            if (scenario in FOCAL_SCENARIOS and h in FOCAL_HORIZONS
                    and name in ['SPARSE_39__DIAG_FROZEN','SPARSE_39__SHRINK_TRAIN_SENSITIVITY']):
                out_file=PROBS/f'{name}__{scenario}__h{h}.npz'
                if out_file.is_file():
                    with np.load(out_file,allow_pickle=False) as z:
                        assert str(z['run_hash'].item())==run_hash
                        assert np.array_equal(z['row_index'],idx)
                        assert np.array_equal(z['p_down_neutral_up'],P)
                else:
                    atomic_npz(out_file,run_hash=np.array(run_hash),row_index=idx,
                        day=day[idx],sequence_id=seq[idx],label_ordered=y,
                        p_down_neutral_up=P,
                        class_names=np.array(['Down','Neutral','Up']),
                        scenario=np.array(scenario),horizon=np.array(h),
                        posterior_latent_extra_variance=np.array(latent_extra))
        print('Finished:',name,scenario)

state=pd.DataFrame(state_rows)
outcome=pd.DataFrame(outcome_rows)
assert len(state)==4*7*4 and len(outcome)==4*7*5*4
atomic_csv(RESULTS/'02_EXACT_STATE_METRICS.csv',state)
atomic_csv(RESULTS/'03_EXACT_OUTCOME_METRICS.csv',outcome)

# Parity of Step 7 full-evidence metrics against Step 6's frozen classifier table.
old=pd.read_csv(SOURCE['step6_outcome_metrics'])
for h in HORIZONS:
    for group in ['POOLED']+[str(d) for d in EVAL_DAYS]:
        a=old[(old['horizon_raw_messages']==h)&(old['day'].astype(str)==group)&
              (old['model']=='ORDERED_PROBIT_SPARSE_DBN_HEAD')]
        b=outcome[(outcome.horizon==h)&(outcome.day==group)&
              (outcome.model=='SPARSE_39__DIAG_FROZEN')&
              (outcome.scenario=='FULL_OBSERVED')]
        assert len(a)==len(b)==1
        for field in ['accuracy','balanced_accuracy','macro_f1','log_loss','brier_multiclass','ece_confidence']:
            assert abs(float(a.iloc[0][field])-float(b.iloc[0][field]))<5e-10,(h,group,field)
print('STEP6↔STEP7 METRIC PARITY PASS: all 5 horizons, pooled and each of 3 days.')

# Contrasts: gains positive if challenger improves; target/mask/horizon matched.
state_p=state[state.day=='POOLED'].pivot(index='scenario',columns='model',values=['state_rmse','state_joint_nll','abs_coverage_error90'])
outcome_p=outcome[outcome.day=='POOLED'].pivot(index=['scenario','horizon'],columns='model',values=['log_loss','brier_multiclass','balanced_accuracy'])
comparisons=[
    ('CROSS_STRUCTURE_DIAG_PRIMARY','AR5__DIAG_FROZEN','SPARSE_39__DIAG_FROZEN'),
    ('COVARIANCE_SENSITIVITY_SPARSE','SPARSE_39__DIAG_FROZEN','SPARSE_39__SHRINK_TRAIN_SENSITIVITY'),
    ('CROSS_STRUCTURE_SHRINK_SENSITIVITY','AR5__SHRINK_TRAIN_SENSITIVITY','SPARSE_39__SHRINK_TRAIN_SENSITIVITY'),
]
contrast_state=[];contrast_outcome=[]
for label,baseline,challenger in comparisons:
    for scenario in MASK_SCENARIOS:
        if scenario!='FULL_OBSERVED':
            contrast_state.append({'contrast':label,'scenario':scenario,
                **{field+'_gain':float(state_p.loc[scenario,(field,baseline)]-
                                      state_p.loc[scenario,(field,challenger)])
                   for field in ['state_rmse','state_joint_nll','abs_coverage_error90']}})
        for h in HORIZONS:
            values={}
            for field in ['log_loss','brier_multiclass']:
                values[field+'_gain']=float(outcome_p.loc[(scenario,h),(field,baseline)]-
                                           outcome_p.loc[(scenario,h),(field,challenger)])
            values['balanced_accuracy_gain']=float(outcome_p.loc[(scenario,h),('balanced_accuracy',challenger)]-
                                                  outcome_p.loc[(scenario,h),('balanced_accuracy',baseline)])
            contrast_outcome.append({'contrast':label,'scenario':scenario,'horizon':h,**values})
state_abl=pd.DataFrame(contrast_state);outcome_abl=pd.DataFrame(contrast_outcome)
atomic_csv(RESULTS/'04_STATE_ABLATIONS.csv',state_abl)
atomic_csv(RESULTS/'05_OUTCOME_ABLATIONS.csv',outcome_abl)

# Day consistency on explicitly small 3-day blocks (not independent tests).
stability=[]
for label,baseline,challenger in comparisons:
    for scenario in MASK_SCENARIOS:
        if scenario=='FULL_OBSERVED':continue
        delta=[]
        for d in EVAL_DAYS:
            a=state[(state.model==baseline)&(state.scenario==scenario)&(state.day==str(d))].iloc[0]
            b=state[(state.model==challenger)&(state.scenario==scenario)&(state.day==str(d))].iloc[0]
            delta.append(float(a.state_rmse-b.state_rmse))
        stability.append({'contrast':label,'scenario':scenario,'days_positive':int(np.sum(np.array(delta)>0)),
                          'days_total':3,'day8_gain':delta[0],'day9_gain':delta[1],
                          'day10_gain':delta[2]})
atomic_csv(RESULTS/'06_DAY_LEVEL_STATE_CONSISTENCY.csv',pd.DataFrame(stability))

summary={'version':VERSION,'status':'EXACT_PARTIAL_EVIDENCE_ASSESSED_RETROSPECTIVELY',
         'run_hash':run_hash,'step6_run_hash':s6['run_hash'],
         'frozen_mask_hash':EXPECTED_MASK_HASH,
         'training_rows':int(len(train_idx)),'retrospective_rows':int(len(eval_idx)),
         'n_state_metrics':len(state),'n_outcome_metrics':len(outcome),
         'probability_columns':['Down','Neutral','Up'],
         'complete_evidence_exact_parity_with_step6':True,
         'graph_and_heads_retrained':False,
         'primary_model':'SPARSE_39__DIAG_FROZEN',
         'sensitivity':'LedoitWolf covariance from D1-D7 residuals, NOT original DBN CPDs',
         'assumption':'exact five-step historical state and noiseless observed contemporaneous state',
         'D8_D10_historically_researcher_blind':False,
         'label_formula_reconstructed_exactly':False,
         'limitations':['Three evaluation days were inspected in earlier project research.',
             'The shrinkage covariance creates correlated residual innovations; it is a sensitivity model, not the original independent-innovation DBN.',
             'Only one-step filtering with exact historical states, not multistep gap smoothing.',
             'Outcomes are publisher labels; accuracy does not establish after-cost profitability.',
             'Do not tune structure, thresholds, Q selection or masks against D8-D10.'],
         'output_files':[p.name for p in sorted(RESULTS.glob('*.csv'))]}
atomic_json(RESULTS/'STEP7_FINAL_SUMMARY.json',summary)

print('\n'+'='*104)
print('STEP 7 COMPLETE — EXACT MISSING-STATE INFERENCE (retrospective, graph unchanged)')
print('='*104)
print('Frozen sparse vs AR5 under diagonal Q: state RMSE improvements (+ = sparse improves):')
display(state_abl[state_abl.contrast=='CROSS_STRUCTURE_DIAG_PRIMARY'][['scenario','state_rmse_gain','state_joint_nll_gain','abs_coverage_error90_gain']].round(6))
print('\nCovariance sensitivity vs frozen sparse diagonal (positive = shrinkage improves):')
display(state_abl[state_abl.contrast=='COVARIANCE_SENSITIVITY_SPARSE'][['scenario','state_rmse_gain','state_joint_nll_gain','abs_coverage_error90_gain']].round(6))
print('\nFocal outcomes h10/h20:')
display(outcome_abl[(outcome_abl.scenario.isin(FOCAL_SCENARIOS))&
                    (outcome_abl.horizon.isin(FOCAL_HORIZONS))][
                   ['contrast','scenario','horizon','log_loss_gain','brier_multiclass_gain','balanced_accuracy_gain']].round(6))
print('\nOutputs:',RESULTS)
print('No trading-profit or identified-causal-effect claim. Runtime:',round(time.time()-T0,1),'s')

Google Drive already mounted.
SELF-TEST PASS: zero/full/partial masks, SPD, diagonal no cross-conditioning, ordinal integration
PREFLIGHT PASS: frozen 39-slot graph and Step6 h10...h100 outputs reproduced bitwise.
Train-only residual rows: 253875  | D8–D10 retrospective rows: 139212
Run hash: 12ac66dc723a267c9082067776d32078dd476241c69d8b69df3e8534b6d8683c
Train-only innovation covariances ready. Shrinkage is sensitivity, NOT a replacement for frozen diagonal DBN.
Finished: AR5__DIAG_FROZEN FULL_OBSERVED
Finished: AR5__DIAG_FROZEN MISS_L1_IMBALANCE
Finished: AR5__DIAG_FROZEN MISS_OFI_PROXY
Finished: AR5__DIAG_FROZEN MISS_BOTH_IMBALANCES
Finished: AR5__DIAG_FROZEN MISS_LIQUIDITY
Finished: AR5__DIAG_FROZEN MISS_RETURN_AND_OFI
Finished: AR5__DIAG_FROZEN HISTORY_ONLY
Finished: AR5__SHRINK_TRAIN_SENSITIVITY FULL_OBSERVED
Finished: AR5__SHRINK_TRAIN_SENSITIVITY MISS_L1_IMBALANCE
Finished: AR5__SHRINK_TRAIN_SENSITIVITY MISS_OFI_PROXY
Finished: AR5__SHRINK_TRAIN_SENSITIVITY MISS_BOTH_IMBALANCE

,scenario,state_rmse_gain,state_joint_nll_gain,abs_coverage_error90_gain
0,MISS_L1_IMBALANCE,0.000000,0.000000,0.000000
1,MISS_OFI_PROXY,0.000332,0.000661,-0.000115
2,MISS_BOTH_IMBALANCES,0.000000,0.000000,0.000000
3,MISS_LIQUIDITY,-0.000207,-0.000438,0.000672
4,MISS_RETURN_AND_OFI,0.004752,0.013698,-0.005984
5,HISTORY_ONLY,0.002209,0.013261,-0.001771



Covariance sensitivity vs frozen sparse diagonal (positive = shrinkage improves):


,scenario,state_rmse_gain,state_joint_nll_gain,abs_coverage_error90_gain
6,MISS_L1_IMBALANCE,0.158365,0.332344,-0.002356
7,MISS_OFI_PROXY,0.248267,0.361488,0.009065
8,MISS_BOTH_IMBALANCES,0.087599,0.347351,-0.001609
9,MISS_LIQUIDITY,-0.000349,0.043611,0.000230
10,MISS_RETURN_AND_OFI,0.063880,0.476854,0.000248
11,HISTORY_ONLY,0.000000,0.522124,-0.000048



Focal outcomes h10/h20:


,contrast,scenario,horizon,log_loss_gain,brier_multiclass_gain,balanced_accuracy_gain
25,CROSS_STRUCTURE_DIAG_PRIMARY,MISS_RETURN_AND_OFI,10,0.008034,0.001527,0.028693
26,CROSS_STRUCTURE_DIAG_PRIMARY,MISS_RETURN_AND_OFI,20,0.005350,-0.000719,0.007611
30,CROSS_STRUCTURE_DIAG_PRIMARY,HISTORY_ONLY,10,0.008104,0.001626,0.031000
31,CROSS_STRUCTURE_DIAG_PRIMARY,HISTORY_ONLY,20,0.006005,0.000196,0.024094
60,COVARIANCE_SENSITIVITY_SPARSE,MISS_RETURN_AND_OFI,10,0.047918,0.033997,0.149928
61,COVARIANCE_SENSITIVITY_SPARSE,MISS_RETURN_AND_OFI,20,0.029641,0.023273,0.097961
65,COVARIANCE_SENSITIVITY_SPARSE,HISTORY_ONLY,10,-0.014945,-0.010293,-0.002633
66,COVARIANCE_SENSITIVITY_SPARSE,HISTORY_ONLY,20,-0.010476,-0.007126,0.000457
95,CROSS_STRUCTURE_SHRINK_SENSITIVITY,MISS_RETURN_AND_OFI,10,0.015055,0.010884,0.026359
96,CROSS_STRUCTURE_SHRINK_SENSITIVITY,MISS_RETURN_AND_OFI,20,0.010098,0.005201,-0.007395



Outputs: /content/drive/MyDrive/CausalLOB/STEP7_MISSING_STATE_READONLY/results
No trading-profit or identified-causal-effect claim. Runtime: 55.9 s


## Exact versus approximate inference check

The primary inference method is exact Gaussian conditioning.  The following cell provides a Monte Carlo comparator using the same frozen Sparse-39 DBN and Days 6–7 only.  It is an implementation benchmark; it does not change the graph or any fitted parameter.


In [ ]:
from pathlib import Path
import os, subprocess, sys
script = Path.cwd() / "Code" / "src" / "05_mc_inference_benchmark.py"
if not script.exists():
    script = Path.cwd().parent / "src" / "05_mc_inference_benchmark.py"
out = Path.cwd() / "Results" / "exact_vs_mc_benchmark.csv"
if not out.parent.exists():
    out = Path.cwd().parents[1] / "Results" / "exact_vs_mc_benchmark.csv"
subprocess.run([
    sys.executable, str(script),
    "--root", os.environ.get("CAUSALLOB_ROOT","/content/drive/MyDrive/CausalLOB"),
    "--out", str(out)
], check=True)
print("Saved:", out)


## M2 interpretation

The final M2 representation has 39 parent slots: 30 own-history lags and 9 validated cross-variable lags.  Structure discovery, parameter learning and inference are separate stages.  Exact partial-observation inference is implemented for the linear-Gaussian DBN, and the equal-history AR(5) model provides the main transition baseline.  Results on historically exposed Days 8–10 are retained only as retrospective execution checks.
